# Tokenization :(

Tokenization is the root cause behind most of the weird behavior you see in LLMs. Seriously, don't skip this.

A lot of things that seem like "LLM problems" are actually **tokenizer problems**:

- **Spelling & string ops** — LLMs can't spell or reverse strings easily because they don't see individual characters, they see tokens (chunks of text).
- **Non-English languages** — Languages like Japanese or Arabic get split into way more tokens, making the model slower and worse at them.
- **Math** — Simple arithmetic fails because numbers get tokenized inconsistently (e.g. "1234" might become ["123", "4"] or ["1", "234"]).
- **Python in GPT-2** — Spaces and indentation were tokenized poorly, making Python code generation harder than it needed to be.
- **Special tokens** — Strings like `<|endoftext|>` can cause the model to stop abruptly because they map to special control tokens.
- **Glitch tokens** — Words like "SolidGoldMagikarp" break the model because they exist in the vocabulary but were barely seen during training.
- **YAML > JSON** — JSON uses more tokens (brackets, quotes) than YAML for the same data, so YAML is cheaper and often works better with LLMs.

Bottom line: if you want to truly understand LLMs, you **must** understand tokenization first.

## Learning map and run order

Work through this notebook in order after a kernel restart: **Unicode and UTF-8 bytes** → **toy byte-level BPE training** → **encoding/decoding** → **regex pre-tokenization** → **special tokens and `tiktoken`** → **SentencePiece** → **your own GPT-4-style tokenizer**.

The early BPE cells intentionally share state. Cell 2 creates `text` and `tokens`; cell 6 defines `get_stats`; cell 8 defines `merge`; cell 11 creates `merges`; and cell 14 creates `vocab`, `decode`, and `encode`. Use **Run All Cells** instead of running a later cell on its own.

# Byte Pair Encoding

Take the input sequence and compress it — find the most frequent pair of adjacent tokens (initially, the tokens are just bytes). Once we identify that most frequently occurring pair, we replace it with a single new token and append it to our vocabulary. Repeat until the vocabulary reaches a chosen target size — the vocab size is the key hyperparameter here, trading off sequence length vs vocab/embedding size.

In [38]:
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/

text= "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."

tokens = text.encode("utf-8")
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 

print('length:', len(text))
print(text)
print('____________')
print('length:', len(tokens))
print('tokens : ', tokens)

length: 533
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
____________
length: 616
tokens :  [239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 1

> The reason why the length of the tokens  are more than the length of text cause the non-ASCII code points convert into multiple bytes (2 up to 4) , while ASCII characters stay 1 byte each

# Byte Pair encoding

We will build a small **byte-level BPE tokenizer**. It starts with a vocabulary of IDs `0..255`, one for each possible byte. Each training step finds the most frequent adjacent pair and gives that pair a new ID.

Keep these roles in mind: `tokens` is the original UTF-8 byte sequence, `ids` is the progressively compressed training sequence, `merges` records how each new token was created, and `vocab` later maps every token ID back to bytes.

In [39]:
chr(101), chr(32)

('e', ' ')

In [40]:
# find the pair of bytes that occure most frequently

def get_stats(ids):
    counts = {}
    for pair in zip(ids , ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)
print(stats)
print(sorted(((v,k) for k,v in stats.items()), reverse=True))

{(239, 188): 1, (188, 181): 1, (181, 239): 1, (239, 189): 6, (189, 142): 1, (142, 239): 1, (189, 137): 1, (137, 239): 1, (189, 131): 1, (131, 239): 1, (189, 143): 1, (143, 239): 1, (189, 132): 1, (132, 239): 1, (189, 133): 1, (133, 33): 1, (33, 32): 2, (32, 240): 3, (240, 159): 15, (159, 133): 7, (133, 164): 1, (164, 240): 1, (133, 157): 1, (157, 240): 1, (133, 152): 1, (152, 240): 1, (133, 146): 1, (146, 240): 1, (133, 158): 1, (158, 240): 1, (133, 147): 1, (147, 240): 1, (133, 148): 1, (148, 226): 1, (226, 128): 12, (128, 189): 1, (189, 32): 1, (159, 135): 7, (135, 186): 1, (186, 226): 1, (128, 140): 6, (140, 240): 6, (135, 179): 1, (179, 226): 1, (135, 174): 1, (174, 226): 1, (135, 168): 1, (168, 226): 1, (135, 180): 1, (180, 226): 1, (135, 169): 1, (169, 226): 1, (135, 170): 1, (170, 33): 1, (159, 152): 1, (152, 132): 1, (132, 32): 1, (32, 84): 1, (84, 104): 1, (104, 101): 6, (101, 32): 20, (32, 118): 1, (118, 101): 3, (101, 114): 6, (114, 121): 2, (121, 32): 2, (32, 110): 2, (110,

In [41]:
# most frequent pair of adjacent tokens (it will be replaced with a new token id, 256+)
top_pair = max(stats , key=stats.get)
top_pair

(101, 32)

In [42]:
# replace the most frequent pair with a new token id (256+)

def merge(ids, pair , idx):
    # in the list of ints (ids), replace all consecutive occurences of pair with the new token idx
    newids = []
    i = 0

    while i <len(ids):
        # if we are not at the very last position AND the pair matches, replace it
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i +=1
    return newids

# example
print(merge([5,6,6,7,9,1],(6,7),99))



[5, 6, 99, 9, 1]


In [43]:
token2 = merge(tokens, top_pair, 256)
print(token2)
print("length: ",len(token2))

[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 159, 135, 169, 226, 128, 140, 240, 159, 135, 170, 33, 32, 240, 159, 152, 132, 32, 84, 104, 256, 118, 101, 114, 121, 32, 110, 97, 109, 256, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 256, 105, 110, 116, 111, 32, 116, 104, 256, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 256, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 256, 111, 117, 103, 104, 116, 32, 116, 111, 32, 226, 128, 156

In [44]:
# making the traning text longer to have more representative token statistics


text = """ In computing, byte-pair encoding (BPE),[1][2] or digram coding,[3] is an algorithm, first described in 1994 by Philip Gage, for encoding strings of text into smaller strings by creating and using a translation table.[4] A slightly modified version of the algorithm is used in large language model tokenizers.

The original version of the algorithm focused on compression. It replaces the highest-frequency pair of bytes with a new byte that was not contained in the initial dataset. A lookup table of the replacements is required to rebuild the initial dataset. The modified version builds "tokens" (units of recognition) that match varying amounts of source text, from single characters (including single digits or single punctuation marks) to whole words (even long compound words).[5][6][7]
Original algorithm

The original BPE algorithm operates by iteratively replacing the most common contiguous sequences of characters in a target text with unused 'placeholder' bytes. The iteration ends when no sequences can be found, leaving the target text effectively compressed. Decompression can be performed by reversing this process, querying known placeholder terms against their corresponding denoted sequence, using a lookup table. In the original paper, this lookup table is encoded and stored alongside the compressed text.
Example

Suppose the data to be encoded is:[8]

aaabdaaabac

The byte pair "aa" occurs most often, so it will be replaced by a byte that is not used in the data, such as "Z". Now there is the following data and replacement table:

ZabdZabac
Z=aa

Then the process is repeated with byte pair "ab", replacing it with "Y":

ZYdZYac
Y=ab
Z=aa

The only literal byte pair left occurs only once, and the encoding might stop here. Alternatively, the process could continue with recursive byte-pair encoding, replacing "ZY" with "X":

XdXac
X=ZY
Y=ab
Z=aa

This data cannot be compressed further by byte-pair encoding because there are no pairs of bytes that occur more than once.

To decompress the data, simply perform the replacements in the reverse order.
Modified algorithm

The original BPE algorithm is modified for use in language modeling, especially for large language models based on neural networks. Compared to the original BPE, the modified BPE does not aim to maximally compress text, but rather, to encode plaintext into "tokens", which are natural numbers.[9] All the unique tokens found in a corpus are listed in a token vocabulary. The token vocabulary can also include some other special tokens, relative to the use case. The size of the token vocabulary, in the case of GPT-3.5 and GPT-4, is 100258 (100000 from BPE algorithm and 258 included as special tokens).[10]

The modified tokenization algorithm initially treats the set of unique characters as 1-character-long n-grams (the initial tokens). Then, successively, the most frequent pair of adjacent tokens is merged into a new, longer n-gram and all instances of the pair are replaced by this new token. This is repeated until a vocabulary of prescribed size is obtained. Note that new words can always be constructed from final vocabulary tokens and initial-set characters.[11]

This modified BPE approach has been extended from spoken language to sign language in recent years.[12]
Example

Suppose we are encoding the previous example of "aaabdaaabac", with a specified vocabulary size of 6, then it would first be encoded as "0, 0, 0, 1, 2, 0, 0, 0, 1, 0, 3" with a vocabulary of "a=0, b=1, d=2, c=3". Then it would proceed as before, and obtain "4, 5, 2, 4, 5, 0, 3" with a vocabulary of "a=0, b=1, d=2, c=3, aa=4, ab=5".

So far this is essentially the same as before. However, if we only had specified a vocabulary size of 5, then the process would stop at vocabulary "a=0, b=1, d=2, c=3, aa=4", so that the example would be encoded as "4, 0, 1, 2, 4, 0, 1, 0, 3". Conversely, if we had specified a vocabulary size of 8, then it would be encoded as "7, 6, 0, 3", with a vocabulary of "a=0, b=1, d=2, c=3, aa=4, ab=5, aaab=6, aaabd=7". This is not maximally compressed, because modified BPE does not aim for maximum compression. Instead, it aims for an encoding that is efficient and practical for language model training.[13]
Byte-level BPE

In the above example, the output of the BPE is a vocabulary, which can be used to encode any text that is written with the letters "abcd". It will not be able to encode text containing other symbols, such as "no". Even giving each of the 26 letters an entry in the vocabulary, since there are many languages in the world using many different scripts, inevitably some symbols would be unencodable by such a vocabulary.

One solution is to replace any unencodable symbol with a special symbol named UNK ("unknown").

The byte-level BPE is another approach. It simply converts the text into UTF-8 first, and treat it as a stream of bytes. This guarantees that any text encoded in UTF-8 can be encoded by the BPE. This has been used in BERT-like models like RoBERTa, BART, and DeBERTa, and GPT-like models like GPT-2.[14][15][16]
 """
tokens = text.encode("utf-8") # raw bytes
tokens = list(map(int ,tokens))

In [45]:
vocab_size = 276
num_merges = vocab_size -256
ids = list(tokens) # copy so we don't destroy the original list

merges = {}
merge_trace = []  # a readable record of what BPE learns at every step

for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats , key=stats.get)
    idx = 256 + i
    merge_trace.append({"step": i + 1, "pair": pair, "count": stats[pair], "new_id": idx})
    print(f"merge {i + 1:>2}: {pair} (count={stats[pair]}) -> {idx}")
    ids = merge(ids , pair,idx)
    merges[pair] = idx

merge  1: (101, 32) (count=139) -> 256
merge  2: (115, 32) (count=93) -> 257
merge  3: (44, 32) (count=92) -> 258
merge  4: (116, 104) (count=89) -> 259
merge  5: (105, 110) (count=77) -> 260
merge  6: (100, 32) (count=72) -> 261
merge  7: (101, 110) (count=64) -> 262
merge  8: (116, 32) (count=56) -> 263
merge  9: (101, 261) (count=46) -> 264
merge 10: (101, 114) (count=43) -> 265
merge 11: (99, 111) (count=42) -> 266
merge 12: (259, 256) (count=42) -> 267
merge 13: (97, 110) (count=40) -> 268
merge 14: (121, 32) (count=40) -> 269
merge 15: (114, 101) (count=40) -> 270
merge 16: (97, 98) (count=39) -> 271
merge 17: (97, 108) (count=37) -> 272
merge 18: (97, 114) (count=36) -> 273
merge 19: (260, 103) (count=32) -> 274
merge 20: (111, 114) (count=32) -> 275


In [46]:
print("token length:" , len(tokens))
print("ids length:" , len(ids))
print(f"compression ratio :{len(tokens) / len(ids):.2f}X")

print("\nFirst five learned merges:")
for row in merge_trace[:5]:
    print(f"step {row['step']:>2}: {row['pair']} occurred {row['count']} times -> token {row['new_id']}")

token length: 5074
ids length: 3923
compression ratio :1.29X

First five learned merges:
step  1: (101, 32) occurred 139 times -> token 256
step  2: (115, 32) occurred 93 times -> token 257
step  3: (44, 32) occurred 92 times -> token 258
step  4: (116, 104) occurred 89 times -> token 259
step  5: (105, 110) occurred 77 times -> token 260


**Note** : The tokenizer is a completely separate, independent module from the LLM.It has its own training dataset of text (Which could be different from that of the LLM) on which you train the vocabulary using the Byte Pair Encoding(BPE) algorithm. It then translates back and forth between raw text and sequences of tokens. The LLM later only ever sees the tokens and never directly deals with any text

In [47]:
# decoding
vocab = {idx:bytes([idx]) for idx in range(256)}
for(p0,p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    # given ids (list of integers) , return Python string
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text


In [48]:
print(decode([128]))

�


In [49]:
# encoding

def encode(text):
    # given a string, return list of integers(the tokens)

    tokens = list(text.encode("utf-8"))
    while len(tokens)>=2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p:merges.get(p,float("inf")))
        if pair not in merges:
            break # nothing else can be merged
        
        idx = merges[pair]
        tokens = merge(tokens , pair, idx)
    return tokens

print(encode("Hello World"))

[72, 101, 108, 108, 111, 32, 87, 275, 108, 100]


Note: `encode` has to replay the merges in the exact order they were learned — the `min` over `merges.get` always picks the pair with the lowest rank (earliest merge) first. Later merges assume the earlier ones have already happened, so doing a fresh frequency count here would tokenize differently than at training time.

In [50]:
print(encode("h"))

[104]


In [51]:
text2 = decode(encode(text))
print(text2 == text)

True


In [52]:
valtext = """Unicode (also known as The Unicode Standard and TUS[1][2]) is a character encoding standard maintained by the Unicode Consortium designed to support the use of text in all of the world's writing systems that can be digitized. Version 17.0[A] defines 159,801 characters and 172 scripts[3] used in various ordinary, literary, academic and technical contexts.

Unicode has largely supplanted the previous environment of myriad incompatible character sets used within different locales and on different computer architectures. The entire repertoire of these sets, plus many additional characters, were merged into the single Unicode set. Unicode is used to encode the vast majority of text on the Internet, including most web pages, and relevant Unicode support has become a common consideration in contemporary software development. Unicode is ultimately capable of encoding more than 1.1 million characters.

The Unicode character repertoire is synchronized with ISO/IEC 10646, each being code-for-code identical with one another. However, The Unicode Standard is more than just a repertoire within which characters are assigned. To aid developers and designers, the standard also provides charts and reference data, as well as annexes explaining concepts germane to various scripts, providing guidance for their implementation. Topics covered by these annexes include character normalization, character composition and decomposition, collation, and directionality.[4]

Unicode encodes 3,790 emoji, with the continued development thereof conducted by the Consortium as a part of the standard.[5] The widespread adoption of Unicode was in large part responsible for the initial popularization of emoji outside of Japan.[citation needed]

Unicode text is processed and stored as binary data using one of several encodings, which define how to translate the standard's abstracted codes for characters into sequences of bytes. The Unicode Standard itself defines three encodings: UTF-8, UTF-16,[a] and UTF-32, though several others exist. UTF-8 is the most widely used by a large margin, in part due to its backwards-compatibility with ASCII.
Origin and development

Unicode was originally designed with the intent of transcending limitations present in all text encodings designed up to that point: each encoding was relied upon for use in its own context, but with no particular expectation of compatibility with any other. Indeed, any two encodings chosen were often totally unworkable when used together, with text encoded in one interpreted as garbage characters by the other. Most encodings had only been designed to facilitate interoperation between a handful of scripts—often primarily between a given script and Latin characters—not between a large number of scripts, and not with all of the scripts supported being treated in a consistent manner.

The philosophy that underpins Unicode seeks to encode the underlying characters—graphemes and grapheme-like units—rather than graphical distinctions considered mere variant glyphs thereof, that are instead best handled by the typeface, through the use of markup, or by some other means. In particularly complex cases, such as the treatment of orthographical variants in Han characters, there is considerable disagreement regarding which differences justify their own encodings, and which are only graphical variants of other characters.

At the most abstract level, Unicode assigns a unique number called a code point to each character. Many issues of visual representation—including size, shape, and style—are intended to be up to the discretion of the software actually rendering the text, such as a web browser or word processor. However, partially with the intent of encouraging rapid adoption, the simplicity of this original model has become somewhat more elaborate over time, and various pragmatic concessions have been made over the course of the standard's development.

The first 256 code points mirror the ISO/IEC 8859-1 standard, with the intent of trivializing the conversion of text already written in Western European scripts. To preserve the distinctions made by different legacy encodings, therefore allowing for conversion between them and Unicode without any loss of information, many characters nearly identical to others, in both appearance and intended function, were given distinct code points. For example, the Halfwidth and Fullwidth Forms block encompasses a full semantic duplicate of the Latin alphabet, because legacy CJK encodings contained both "fullwidth" (matching the width of CJK characters) and "halfwidth" (matching ordinary Latin script) characters.
History

The origins of Unicode can be traced back to the 1980s, to a group of individuals with connections to Xerox's Character Code Standard (XCCS).[6] In 1987, Xerox employee Joe Becker, along with Apple employees Lee Collins and Mark Davis, started investigating the practicalities of creating a universal character set.[7] With additional input from Peter Fenwick and Dave Opstad,[6] Becker published a draft proposal for an "international/multilingual text character encoding system in August 1988, tentatively called Unicode". He explained that "the name 'Unicode' is intended to suggest a unique, unified, universal encoding".[6]

In this document, entitled Unicode 88, Becker outlined a scheme using 16-bit characters:[6]

    Unicode is intended to address the need for a workable, reliable world text encoding. Unicode could be roughly described as "wide-body ASCII" that has been stretched to 16 bits to encompass the characters of all the world's living languages. In a properly engineered design, 16 bits per character are more than sufficient for this purpose.

This design decision was made based on the assumption that only scripts and characters in "modern" use would require encoding:[6]

    Unicode gives higher priority to ensuring utility for the future than to preserving past antiquities. Unicode aims in the first instance at the characters published in the modern text (e.g. in the union of all newspapers and magazines printed in the world in 1988), whose number is undoubtedly far below 214 = 16,384. Beyond those modern-use characters, all others may be defined to be obsolete or rare; these are better candidates for private use registration than for congesting the public list of generally useful Unicode.

In early 1989, the Unicode working group expanded to include Ken Whistler and Mike Kernaghan of Metaphor, Karen Smith-Yoshimura and Joan Aliprand of Research Libraries Group, and Glenn Wright of Sun Microsystems. The Research Libraries Group had an existing solution for East Asian character sets, which became one of the inputs to the Unicode character set.[7] In 1990, Michel Suignard and Asmus Freytag of Microsoft and NeXT's Rick McGowan had also joined the group. By the end of 1990, most of the work of remapping existing standards had been completed, and a final review draft of Unicode was ready.

The Unicode Consortium was incorporated in California on 3 January 1991,[8] and the first volume of The Unicode Standard was published that October. The second volume, now adding Han ideographs, was published in June 1992.

In 1996, a surrogate character mechanism was implemented in Unicode 2.0, so that Unicode was no longer restricted to 16 bits. This increased the Unicode codespace to over a million code points, which allowed for the encoding of many historic scripts, such as Egyptian hieroglyphs, and thousands of rarely used or obsolete characters that had not been anticipated for inclusion in the standard. Among these characters are various rarely used CJK characters—many mainly being used in proper names, making them far more necessary for a universal encoding than the original Unicode architecture envisioned.[9]
Unicode Consortium
Main article: Unicode Consortium

The Unicode Consortium is a non-profit organization that coordinates Unicode's development. Full members include most of the main computer software and hardware companies (and few others) with any interest in text-processing standards, including Adobe, Apple, Google, IBM, Meta (previously as Facebook), Microsoft, Netflix, and SAP.[10]

Over the years several countries or government agencies have been members of the Unicode Consortium.[10]

The Consortium has the ambitious goal of eventually replacing existing character encoding schemes with Unicode and its standard Unicode Transformation Format (UTF) schemes, as many of the existing schemes are limited in size and scope and are incompatible with multilingual environments.

The Unicode Bulldog Award is given to people deemed to be influential in Unicode's development, with recipients including Tatsuo Kobayashi, Thomas Milo, Roozbeh Pournader, Ken Lunde, and Michael Everson.[11]
Scripts covered
Main article: Script (Unicode)
Many modern applications can render a substantial subset of the many scripts in Unicode, as demonstrated by this screenshot from the OpenOffice.org application.

As of September 2025, a total of 172[12] scripts (alphabets, abugidas and syllabaries) are included in Unicode, covering most major writing systems in use today.[13][14] There are still scripts that are not yet encoded, particularly those mainly used in historical, liturgical, and academic contexts. Further additions of characters to the already encoded scripts, as well as symbols, in particular for mathematics and music also occur.
Proposals for adding scripts

The Unicode Roadmap Committee (Michael Everson, Rick McGowan, Ken Whistler, V.S. Umamaheswaran)[15] maintain the list of scripts that are candidates or potential candidates for encoding and their tentative code block assignments on the Unicode Roadmap[16] page of the Unicode Consortium website. For some scripts on the Roadmap, such as Jurchen and Khitan large script, encoding proposals have been made and they are working their way through the approval process. For other scripts, such as Numidian and Rongorongo, no proposal has yet been made, and they await agreement on character repertoire and other details from the user communities involved.

Some modern invented scripts which have not yet been included in Unicode (e.g., Tengwar) or which do not qualify for inclusion in Unicode due to lack of real-world use (e.g., Klingon) are listed in the ConScript Unicode Registry, along with unofficial but widely used private use area code assignments.

There is also a Medieval Unicode Font Initiative focused on special Latin medieval characters. Part of these proposals has been already included in Unicode.

The Script Encoding Initiative (SEI),[17] a project created by Deborah Anderson at the University of California, Berkeley, was founded in 2002 with the goal of funding proposals for scripts not yet encoded in the standard. Now run by Anushah Hossain, SEI has become a major source of proposed additions to the standard in recent years.[18] Although SEI collaborates with the Unicode Consortium and the ISO/IEC 10646 standards process, it operates independently, supporting the technical, linguistic, and historical research needed to prepare formal proposals. SEI maintains a database of scripts that have yet to be encoded in the Unicode Standard on the project's website.[19]
Versions

The Unicode Consortium together with the ISO have developed a shared repertoire following the initial publication of The Unicode Standard: Unicode and the ISO's Universal Coded Character Set (UCS) use identical character names and code points. However, the Unicode versions do differ from their ISO equivalents in two significant ways.

While the UCS is a simple character map, Unicode specifies the rules, algorithms, and properties necessary to achieve interoperability between different platforms and languages. Thus, The Unicode Standard includes more information, covering in-depth topics such as bitwise encoding, collation, and rendering. It also provides a comprehensive catalog of character properties, including those needed for supporting bidirectional text, as well as visual charts and reference data sets to aid implementers. Previously, The Unicode Standard was sold as a print volume containing the complete core specification, standard annexes,[note 1] and code charts. However, version 5.0, published in 2006, was the last version printed this way. Starting with version 5.2, only the core specification, published as a print-on-demand paperback, may be purchased.[20] The full text, on the other hand, is published as a free PDF on the Unicode website.

A practical reason for this publication method highlights the second significant difference between the UCS and Unicode—the frequency with which updated versions are released and new characters added. The Unicode Standard has regularly released annual expanded versions, occasionally with more than one version released in a calendar year and with rare cases where the scheduled release had to be postponed. For instance, in April 2020, a month after version 13.0 was published, the Unicode Consortium announced they had changed the intended release date for version 14.0, pushing it back six months to September 2021 due to the COVID-19 pandemic.

Thus far, the following versions of The Unicode Standard have been published. Update versions, which do not include any changes to character repertoire, are signified by the third number (e.g., "version 4.0.1") and are omitted in the table below.[21]
Unicode version history and notable changes to characters and scripts Ver­sion 	Date 	Publication
(book, text) 	UCS edition 	Total 	Details
Scripts 	Characters[b]
1.0.0[22] 	October 1991 	ISBN 0-201-56788-1 (vol. 1) 	—N/a 	24 	7129 	Initial scripts covered: Arabic, Armenian, Bengali, Bopomofo, Cyrillic, Devanagari, Georgian, Greek and Coptic, Gujarati, Gurmukhi, Hangul, Hebrew, Hiragana, Kannada, Katakana, Lao, Latin, Malayalam, Odia, Tamil, Telugu, Thai, and Tibetan
1.0.1[23] 	June 1992 	ISBN 0-201-60845-6 (vol. 2) 	25 	28327+21204
−6 	The initial 20,902 CJK Unified Ideographs
1.1[24] 	June 1993 	—N/a 	ISO/IEC 10646-1:1993

[c]
	24 	34168+5963
−9 	33 reclassified as control characters. 4,306 Hangul syllables, Tibetan removed
2.0[25] 	July 1996 	ISBN 0-201-48345-9 	25 	38885+11373
−6656 	Original set of Hangul syllables removed, new set of 11,172 Hangul syllables added at new location, Tibetan added back in a new location and with a different character repertoire, Surrogate character mechanism defined, Plane 15 and Plane 16 private use area allocated
2.1[26] 	May 1998 	—N/a 	38887+2
	U+20AC € EURO SIGN, U+FFFC ￼ OBJECT REPLACEMENT CHARACTER[26]
3.0[27] 	September 1999 	ISBN 0-201-61633-5 	ISO/IEC 10646-1:2000 	38 	49194+10307
	Cherokee, Geʽez, Khmer, Mongolian, Burmese, Ogham, runes, Sinhala, Syriac, Thaana, Canadian Aboriginal syllabics, and Yi Syllables, Braille patterns
3.1[28] 	March 2001 	—N/a 	ISO/IEC 10646-1:2000[d]ISO/IEC 10646-2:2001 	41 	94140+44946
	Deseret, Gothic and Old Italic, sets of symbols for Western and Byzantine music, 42,711 additional CJK Unified Ideographs
3.2[29] 	March 2002 	45 	95156+1016
	Philippine scripts (Buhid, Hanunoo, Tagalog, and Tagbanwa), mathematical symbols
4.0[30] 	April 2003 	ISBN 0-321-18578-1 	ISO/IEC 10646:2003

[e]
	52 	96382+1226
	Cypriot syllabary, Limbu, Linear B, Osmanya, Shavian, Tai Le, and Ugaritic, Hexagram symbols
4.1[31] 	March 2005 	—N/a 	59 	97655+1273
	Buginese, Glagolitic, Kharosthi, New Tai Lue, Old Persian, Sylheti Nagri, and Tifinagh, Coptic disunified from Greek, ancient Greek numbers and musical symbols, first named character sequences were introduced.[32]
5.0[33] 	July 2006 	ISBN 0-321-48091-0 	64 	99024+1369
	Balinese, cuneiform, N'Ko, ʼPhags-pa, Phoenician[34]
5.1[35] 	April 2008 	—N/a 	75 	100648+1624
	Carian, Cham, Kayah Li, Lepcha, Lycian, Lydian, Ol Chiki, Rejang, Saurashtra, Sundanese, and Vai, sets of symbols for the Phaistos Disc, Mahjong tiles, Domino tiles, additions to Burmese, Scribal abbreviations, U+1E9E ẞ LATIN CAPITAL LETTER SHARP S
5.2[36] 	October 2009 	ISBN 978-1-936213-00-9 	90 	107296+6648
	Avestan, Bamum, Gardiner's sign list of Egyptian hieroglyphs, Imperial Aramaic, Inscriptional Pahlavi, Inscriptional Parthian, Javanese, Kaithi, Lisu, Meetei Mayek, Old South Arabian, Old Turkic, Samaritan, Tai Tham and Tai Viet, additional CJK Unified Ideographs, Jamo for Old Hangul, Vedic Sanskrit
6.0[37] 	October 2010 	ISBN 978-1-936213-01-6 	ISO/IEC 10646:2010

[f]
	93 	109384+2088
	Batak, Brahmi, Mandaic, playing card symbols, transport and map symbols, alchemical symbols, emoticons and emoji,[38] additional CJK Unified Ideographs
6.1[39] 	January 2012 	ISBN 978-1-936213-02-3 	ISO/IEC 10646:2012

[g]
	100 	110116+732
	Chakma, Meroitic cursive, Meroitic hieroglyphs, Miao, Sharada, Sora Sompeng, and Takri
6.2[40] 	September 2012 	ISBN 978-1-936213-07-8 	110117+1
	U+20BA ₺ TURKISH LIRA SIGN
6.3[41] 	September 2013 	ISBN 978-1-936213-08-5 	110122+5
	5 bidirectional formatting characters
7.0[42] 	June 2014 	ISBN 978-1-936213-09-2 	123 	112956+2834
	Bassa Vah, Caucasian Albanian, Duployan, Elbasan, Grantha, Khojki, Khudawadi, Linear A, Mahajani, Manichaean, Mende Kikakui, Modi, Mro, Nabataean, Old North Arabian, Old Permic, Pahawh Hmong, Palmyrene, Pau Cin Hau, Psalter Pahlavi, Siddham, Tirhuta, Warang Citi, and dingbats
8.0[43] 	June 2015 	ISBN 978-1-936213-10-8 	ISO/IEC 10646:2014

[h]
	129 	120672+7716
	Ahom, Anatolian hieroglyphs, Hatran, Multani, Old Hungarian, SignWriting, additional CJK Unified Ideographs, lowercase letters for Cherokee, 5 emoji skin tone modifiers
9.0[46] 	June 2016 	ISBN 978-1-936213-13-9 	135 	128172+7500
	Adlam, Bhaiksuki, Marchen, Newa, Osage, Tangut, 72 emoji[47]
10.0[48] 	June 2017 	ISBN 978-1-936213-16-0 	ISO/IEC 10646:2017

[i]
	139 	136690+8518
	Zanabazar Square, Soyombo, Masaram Gondi, Nüshu, hentaigana, 7,494 CJK Unified Ideographs, 56 emoji, U+20BF ₿ BITCOIN SIGN
11.0[49] 	June 2018 	ISBN 978-1-936213-19-1 	146 	137374+684
	Dogra, Georgian Mtavruli capital letters, Gunjala Gondi, Hanifi Rohingya, Indic Siyaq Numbers, Makasar, Medefaidrin, Old Sogdian and Sogdian, Maya numerals, 5 CJK Unified Ideographs, symbols for xiangqi and star ratings, 145 emoji
12.0[50] 	March 2019 	ISBN 978-1-936213-22-1 	150 	137928+554
	Elymaic, Nandinagari, Nyiakeng Puachue Hmong, Wancho, Miao script, hiragana and katakana small letters, Tamil historic fractions and symbols, Lao letters for Pali, Latin letters for Egyptological and Ugaritic transliteration, hieroglyph format controls, 61 emoji
12.1[51] 	May 2019 	ISBN 978-1-936213-25-2 	137929+1
	U+32FF ㋿ SQUARE ERA NAME REIWA
13.0[52] 	March 2020 	ISBN 978-1-936213-26-9 	ISO/IEC 10646:2020

[53]
	154 	143859+5930
	Chorasmian, Dhives Akuru, Khitan small script, Yezidi, 4,969 CJK ideographs, Arabic script additions used to write Hausa, Wolof, and other African languages, additions used to write Hindko and Punjabi in Pakistan, Bopomofo additions used for Cantonese, Creative Commons license symbols, graphic characters for compatibility with teletext and home computer systems, 55 emoji
14.0[54] 	September 2021 	ISBN 978-1-936213-29-0 	159 	144697+838
	Toto, Cypro-Minoan, Vithkuqi, Old Uyghur, Tangsa, extended IPA, Arabic script additions for use in languages across Africa and in Iran, Pakistan, Malaysia, Indonesia, Java, and Bosnia, additions for honorifics and Quranic use, additions to support languages in North America, the Philippines, India, and Mongolia, U+20C0 ⃀ SOM SIGN, Znamenny musical notation, 37 emoji
15.0[55] 	September 2022 	ISBN 978-1-936213-32-0 	161 	149186+4489
	Kawi and Mundari, 20 emoji, 4,192 CJK ideographs, control characters for Egyptian hieroglyphs
15.1[56] 	September 2023 	ISBN 978-1-936213-33-7 	149813+627
	Additional CJK ideographs
16.0[57] 	September 2024 	ISBN 978-1-936213-34-4 		168 	154998+5185
	Garay, Gurung Khema, Kirat Rai, Ol Onal, Sunuwar, Todhri, Tulu-Tigalari, 7 emoji, 3,995 Egyptian Hieroglyphs
17.0[58] 	September 2025 	ISBN 978-1-936213-35-1 		172 	159801+4803
	Beria Erfe, Tai Yo, Sidetic, Tolong Siki, U+20C1 ⃁ SAUDI RIYAL SIGN, 7 emoji, 4,316 CJK unified ideographs

    A large amount of documentation for Windows incorrectly uses the term "Unicode" to mean only the UTF-16 encoding.
    The total number of graphic and format characters, excluding private use characters, control characters, noncharacters, and surrogate code points).
        2.0 added Amendments 5, 6, and 72.1 added two characters from Amendment 18.
    3.2 added Amendment 1.
        4.1 added Amendment 15.0 added Amendment 2 as well as four characters from Amendment 35.1 added Amendment 45.2 added Amendments 5 and 6
    Plus the Indian rupee sign
        6.2 added the Turkish lira sign6.3 added five additional characters7.0 added Amendments 1 and 2 as well as the ruble sign
    Plus Amendment 1, as well as the Lari sign, nine CJK unified ideographs, and 41 emoji;[44]
    9.0 added Amendment 2, as well as Adlam, Newa, Japanese TV symbols, and 74 emoji and symbols.[45]
        Plus 56 emoji, 285 hentaigana characters, and 3 Zanabazar Square characters11.0 added 46 Mtavruli Georgian capital letters, 5 CJK unified ideographs, and 66 emoji12.0 added 62 additional characters.

Architecture and terminology
See also: Universal Character Set characters
Codespace and code points

The Unicode Standard defines a codespace:[59] a sequence of integers called code points[60] in the range from 0 to 1114111, notated according to the standard as U+0000–U+10FFFF.[61] The codespace is a systematic, architecture-independent representation of The Unicode Standard; actual text is processed as binary data via one of several Unicode encodings, such as UTF-8.

In this normative notation, the two-character prefix U+ always precedes a written code point, and the code points themselves are written as hexadecimal numbers.[note 2] At least four hexadecimal digits are always written, with leading zeros prepended as needed. For example, the code point U+00F7 ÷ DIVISION SIGN is padded with two leading zeros, but U+13254 𓉔 EGYPTIAN HIEROGLYPH O004 () is not padded.[63]

There are a total of 1112064 valid code points within the codespace.[64] This number arises from the limitations of the UTF-16 character encoding, which can encode the 216 code points in the range U+0000 through U+FFFF except for the 211 code points in the range U+D800 through U+DFFF, which are used as surrogate pairs to encode the 220 code points in the range U+10000 through U+10FFFF.
Code planes and blocks
Main article: Plane (Unicode)

The Unicode codespace is divided into 17 planes, numbered 0 to 16. Plane 0 is the Basic Multilingual Plane (BMP), and contains the most commonly used characters. All code points in the BMP are accessed as a single code unit in UTF-16 encoding and can be encoded in one, two or three bytes in UTF-8. Code points in planes 1 through 16 (the supplementary planes) are accessed as surrogate pairs in UTF-16 and encoded in four bytes in UTF-8.

Within each plane, characters are allocated within named blocks of related characters. The size of a block is always a multiple of 16, and is often a multiple of 128, but is otherwise arbitrary. Characters required for a given script may be spread out over several different, potentially disjunct blocks within the codespace.
General Category property

Each code point is assigned a classification, listed as the code point's General Category property. Here, at the uppermost level code points are categorized as one of Letter, Mark, Number, Punctuation, Symbol, Separator, or Other. Under each category, each code point is then further subcategorized. In most cases, other properties must be used to adequately describe all the characteristics of any given code point.
General Category (Unicode Character Property)[a]

    vte

Value	Category Major, minor	Basic type[b]	Character assigned[b]	Count[c]
(as of 17.0)	Remarks
  					
L, Letter; LC, Cased Letter (Lu, Ll, and Lt only)[d]
Lu	Letter, uppercase	Graphic	Character 	1,886	
Ll	Letter, lowercase	Graphic	Character 	2,283	
Lt	Letter, titlecase	Graphic	Character 	31	Digraphs consisting of an uppercase letter followed by a lowercase letter (e.g., ǅ, ǈ, ǋ, and ǲ)
Lm	Letter, modifier	Graphic	Character 	410	A modifier letter
Lo	Letter, other	Graphic	Character 	141,062	An ideograph or a letter in a unicase alphabet
M, Mark
Mn	Mark, nonspacing	Graphic	Character 	2,059	
Mc	Mark, spacing combining	Graphic	Character 	471	
Me	Mark, enclosing	Graphic	Character 	13	
N, Number
Nd	Number, decimal digit	Graphic	Character 	770	All these, and only these, have Numeric Type = De[e]
Nl	Number, letter	Graphic	Character 	239	Numerals composed of letters or letterlike symbols (e.g., Roman numerals)
No	Number, other	Graphic	Character 	915	E.g., vulgar fractions, superscript and subscript digits, vigesimal digits
P, Punctuation
Pc	Punctuation, connector	Graphic	Character 	10	Includes spacing underscore characters such as "_", and other spacing tie characters. Unlike other punctuation characters, these may be classified as "word" characters by regular expression libraries.[f]
Pd	Punctuation, dash	Graphic	Character 	27	Includes several hyphen characters
Ps	Punctuation, open	Graphic	Character 	79	Opening bracket characters
Pe	Punctuation, close	Graphic	Character 	77	Closing bracket characters
Pi	Punctuation, initial quote	Graphic	Character 	12	Opening quotation mark. Does not include the ASCII "neutral" quotation mark. May behave like Ps or Pe depending on usage
Pf	Punctuation, final quote	Graphic	Character 	10	Closing quotation mark. May behave like Ps or Pe depending on usage
Po	Punctuation, other	Graphic	Character 	641	
S, Symbol
Sm	Symbol, math	Graphic	Character 	960	Mathematical symbols (e.g., +, −, =, ×, ÷, √, ∊, ≠). Does not include parentheses and brackets, which are in categories Ps and Pe. Also does not include !, *, -, or /, which despite frequent use as mathematical operators, are primarily considered to be "punctuation".
Sc	Symbol, currency	Graphic	Character 	64	Currency symbols
Sk	Symbol, modifier	Graphic	Character 	125	
So	Symbol, other	Graphic	Character 	7,468	
Z, Separator
Zs	Separator, space	Graphic	Character 	17	Includes the space, but not TAB, CR, or LF, which are Cc
Zl	Separator, line	Format	Character 	1	Only U+2028 LINE SEPARATOR (LSEP)
Zp	Separator, paragraph	Format	Character 	1	Only U+2029 PARAGRAPH SEPARATOR (PSEP)
C, Other
Cc	Other, control	Control	Character 	65 (will never change)[e]	No name,[g] <control>
Cf	Other, format	Format	Character 	170	Includes the soft hyphen, joining control characters (ZWNJ and ZWJ), control characters to support bidirectional text, and language tag characters
Cs	Other, surrogate	Surrogate	Not (only used in UTF-16) 	2,048 (will never change)[e]	No name,[g] <surrogate>
Co	Other, private use	Private-use	Character (but no interpretation specified) 	137,468 total (will never change)[e] (6,400 in BMP, 131,068 in Planes 15–16)	No name,[g] <private-use>
Cn	Other, not assigned	Noncharacter	Not 	66 (will not change unless the range of Unicode code points is expanded)[e]	No name,[g] <noncharacter>
Reserved	Not 	814,664	No name,[g] <reserved>

    "Table 4-4: General Category". The Unicode Standard. Unicode Consortium. September 2025.
    "Table 2-3: Types of code points". The Unicode Standard. Unicode Consortium. September 2025.
    "DerivedGeneralCategory.txt". The Unicode Consortium. 2025-07-24.
    "5.7.1 General Category Values". UTR #44: Unicode Character Database. Unicode Consortium. 2024-08-27.
    Unicode Character Encoding Stability Policies: Property Value Stability Stability policy: Some gc groups will never change. gc=Nd corresponds with Numeric Type=De (decimal).
    "Annex C: Compatibility Properties (§ word)". Unicode Regular Expressions. Version 23. Unicode Consortium. 2022-02-08. Unicode Technical Standard #18.
    "Table 4-9: Construction of Code Point Labels". The Unicode Standard. Unicode Consortium. September 2025. A Code Point Label may be used to identify a nameless code point. E.g. <control-hhhh>, <control-0088>. The Name remains blank, which can prevent inadvertently replacing, in documentation, a Control Name with a true Control code. Unicode also uses <not a character> for <noncharacter>.

The 1024 points in the range U+D800–U+DBFF are known as high-surrogate code points, and code points in the range U+DC00–U+DFFF (1024 code points) are known as low-surrogate code points. A high-surrogate code point followed by a low-surrogate code point forms a surrogate pair in UTF-16 in order to represent code points greater than U+FFFF. In principle, these code points cannot otherwise be used, though in practice this rule is often ignored, especially when not using UTF-16.

A small set of code points are guaranteed never to be assigned to characters, although third-parties may make independent use of them at their discretion. There are 66 of these noncharacters: U+FDD0–U+FDEF and the last two code points in each of the 17 planes (e.g. U+FFFE, U+FFFF, U+1FFFE, U+1FFFF, ..., U+10FFFE, U+10FFFF). The set of noncharacters is stable, and no new noncharacters will ever be defined.[65] Like surrogates, the rule that these cannot be used is often ignored, although the operation of the byte order mark assumes that U+FFFE will never be the first code point in a text. The exclusion of surrogates and noncharacters leaves 1111998 code points available for use.

Private use code points are considered to be assigned, but they intentionally have no interpretation specified by The Unicode Standard[66] such that any interchange of such code points requires an independent agreement between the sender and receiver as to their interpretation. There are three private use areas in the Unicode codespace:

    Private Use Area: U+E000–U+F8FF (6400 characters),
    Supplementary Private Use Area-A: U+F0000–U+FFFFD (65534 characters),
    Supplementary Private Use Area-B: U+100000–U+10FFFD (65534 characters).

Graphic characters are those defined by The Unicode Standard to have particular semantics, either having a visible glyph shape or representing a visible space. As of Unicode 17.0, there are 159629 graphic characters.

Format characters are characters that do not have a visible appearance but may have an effect on the appearance or behavior of neighboring characters. For example, U+200C ZERO WIDTH NON-JOINER and U+200D ZERO WIDTH JOINER may be used to change the default shaping behavior of adjacent characters (e.g. to inhibit ligatures or request ligature formation). There are 172 format characters in Unicode 17.0.

65 code points, the ranges U+0000–U+001F and U+007F–U+009F, are reserved as control codes, corresponding to the C0 and C1 control codes as defined in ISO/IEC 6429. U+0009 TAB, U+000A LINE FEED, and U+000D CARRIAGE RETURN are widely used in texts using Unicode. In a phenomenon known as mojibake, the C1 code points are improperly decoded according to the Windows-1252 codepage, previously widely used in Western European contexts.

Together, graphic, format, control code, and private use characters are collectively referred to as assigned characters. Reserved code points are those code points that are valid and available for use, but have not yet been assigned. As of Unicode 17.0, there are 814664 reserved code points.
Abstract characters
Further information: Universal Character Set characters § Characters, grapheme clusters and glyphs

The set of graphic and format characters defined by Unicode does not correspond directly to the repertoire of abstract characters representable under Unicode. Unicode encodes characters by associating an abstract character with a particular code point.[67] However, not all abstract characters are encoded as a single Unicode character, and some abstract characters may be represented in Unicode by a sequence of two or more characters. For example, a Latin small letter "i" with an ogonek, a dot above, and an acute accent, which is required in Lithuanian, is represented by the character sequence U+012F; U+0307; U+0301. Unicode maintains a list of uniquely named character sequences for abstract characters that are not directly encoded in Unicode.[68]

All assigned characters have a unique and immutable name by which they are identified. This immutability has been guaranteed since version 2.0 of The Unicode Standard by its Name Stability policy.[65] In cases where a name is seriously defective and misleading, or has a serious typographical error, a formal alias may be defined that applications are encouraged to use in place of the official character name. For example, U+A015 ꀕ YI SYLLABLE WU has the formal alias YI SYLLABLE ITERATION MARK, and U+FE18 ︘ PRESENTATION FORM FOR VERTICAL RIGHT WHITE LENTICULAR BRAKCET (sic) has the formal alias PRESENTATION FORM FOR VERTICAL RIGHT WHITE LENTICULAR BRACKET.[69]
Precomposed vis-à-vis composite characters

Unicode includes a mechanism for modifying characters that greatly extends the supported repertoire of glyphs. This covers the use of combining diacritical marks that may be added after the base character by the user. Multiple combining diacritics may be simultaneously applied to the same character. Unicode also contains precomposed versions of most letter/diacritic combinations in normal use. These make the conversion to and from legacy encodings simpler, and allow applications to use Unicode as an internal text format without having to implement combining characters. For example, é can be represented in Unicode as U+0065 e LATIN SMALL LETTER E followed by U+0301 ◌́ COMBINING ACUTE ACCENT, and equivalently as the precomposed character U+00E9 é LATIN SMALL LETTER E WITH ACUTE. Thus, users often have multiple equivalent ways of encoding the same character. The mechanism of canonical equivalence within The Unicode Standard ensures the practical interchangeability of these equivalent encodings.

An example of this arises with the Korean alphabet Hangul: Unicode provides a mechanism for composing Hangul syllables from their individual Hangul Jamo subcomponents. However, it also provides 11172 combinations of precomposed syllables made from the most common jamo.

CJK characters presently only have codes for uncomposable radicals and precomposed forms. Most Han characters have either been intentionally composed from, or reconstructed as compositions of, simpler orthographic elements called radicals, so in principle Unicode could have enabled their composition as it did with Hangul. While this could have greatly reduced the number of required code points, as well as allowing the algorithmic synthesis of many arbitrary new characters, the complexities of character etymologies and the post-hoc nature of radical systems add immense complexity to the proposal. Indeed, attempts to design CJK encodings on the basis of composing radicals have been met with difficulties resulting from the reality that Chinese characters do not decompose as simply or as regularly as Hangul does.

The CJK Radicals Supplement block is assigned to the range U+2E80–U+2EFF, and the Kangxi radicals are assigned to U+2F00–U+2FDF. The Ideographic Description Sequences block covers the range U+2FF0–U+2FFB, but The Unicode Standard warns against using its characters as an alternate representation for characters encoded elsewhere:

    This process is different from a formal encoding of an ideograph. There is no canonical description of unencoded ideographs; there is no semantic assigned to described ideographs; there is no equivalence defined for described ideographs. Conceptually, ideographic descriptions are more akin to the English phrase "an 'e' with an acute accent on it" than to the character sequence <U+0065, U+0301>.

Ligatures
The Devanāgarī ddhrya-ligature (द् + ध् + र् + य = द्ध्र्य) of JanaSanskritSans[70]
The Arabic lām-alif ligature (ل ‎+‎ ا ‎=‎ لا)

Many scripts, including Arabic and Devanāgarī, have special orthographic rules that require certain combinations of letterforms to be combined into special ligature forms. The rules governing ligature formation can be quite complex, requiring special script-shaping technologies such as ACE (Arabic Calligraphic Engine by DecoType in the 1980s and used to generate all the Arabic examples in the printed editions of The Unicode Standard), which became the proof of concept for OpenType (by Adobe and Microsoft), Graphite (by SIL International), or AAT (by Apple).

Instructions are also embedded in fonts to tell the operating system how to properly output different character sequences. A simple solution to the placement of combining marks or diacritics is assigning the marks a width of zero and placing the glyph itself to the left or right of the left sidebearing (depending on the direction of the script they are intended to be used with). A mark handled this way will appear over whatever character precedes it, but will not adjust its position relative to the width or height of the base glyph; it may be visually awkward and it may overlap some glyphs. Real stacking is impossible but can be approximated in limited cases (for example, Thai top-combining vowels and tone marks can just be at different heights to start with). Generally, this approach is only effective in monospaced fonts but may be used as a fallback rendering method when more complex methods fail.
Standardized subsets

Several subsets of Unicode are standardized: Microsoft Windows since Windows NT 4.0 supports WGL-4 with 657 characters, which is considered to support all contemporary European languages using the Latin, Greek, or Cyrillic script. Other standardized subsets of Unicode include the Multilingual European Subsets:[71] MES-1 (Latin scripts only; 335 characters), MES-2 (Latin, Greek, and Cyrillic; 1062 characters)[72] and MES-3A & MES-3B (two larger subsets, not shown here). MES-2 includes every character in MES-1 and WGL-4.

The standard DIN 91379[73] specifies a subset of Unicode letters, special characters, and sequences of letters and diacritic signs to allow the correct representation of names and to simplify data exchange in Europe. This standard supports all of the official languages of all European Union countries, as well as the German minority languages and the official languages of Iceland, Liechtenstein, Norway, and Switzerland. To allow the transliteration of names in other writing systems to the Latin script according to the relevant ISO standards, all necessary combinations of base letters and diacritic signs are provided.
WGL-4, MES-1 and MES-2 Row	Cells	Range(s)
00 	20–7E 	Basic Latin (00–7F)
A0–FF 	Latin-1 Supplement (80–FF)
01 	00–13, 14–15, 16–2B, 2C–2D, 2E–4D, 4E–4F, 50–7E, 7F 	Latin Extended-A (00–7F)
8F, 92, B7, DE-EF, FA–FF 	Latin Extended-B (80–FF ...)
02 	18–1B, 1E–1F 	Latin Extended-B (... 00–4F)
59, 7C, 92 	IPA Extensions (50–AF)
BB–BD, C6, C7, C9, D6, D8–DB, DC, DD, DF, EE 	Spacing Modifier Letters (B0–FF)
03 	74–75, 7A, 7E, 84–8A, 8C, 8E–A1, A3–CE, D7, DA–E1 	Greek (70–FF)
04 	00–5F, 90–91, 92–C4, C7–C8, CB–CC, D0–EB, EE–F5, F8–F9 	Cyrillic (00–FF)
1E 	02–03, 0A–0B, 1E–1F, 40–41, 56–57, 60–61, 6A–6B, 80–85, 9B, F2–F3 	Latin Extended Additional (00–FF)
1F 	00–15, 18–1D, 20–45, 48–4D, 50–57, 59, 5B, 5D, 5F–7D, 80–B4, B6–C4, C6–D3, D6–DB, DD–EF, F2–F4, F6–FE 	Greek Extended (00–FF)
20 	13–14, 15, 17, 18–19, 1A–1B, 1C–1D, 1E, 20–22, 26, 30, 32–33, 39–3A, 3C, 3E, 44, 4A 	General Punctuation (00–6F)
7F, 82 	Superscripts and Subscripts (70–9F)
A3–A4, A7, AC, AF 	Currency Symbols (A0–CF)
21 	05, 13, 16, 22, 26, 2E 	Letterlike Symbols (00–4F)
5B–5E 	Number Forms (50–8F)
90–93, 94–95, A8 	Arrows (90–FF)
22 	00, 02, 03, 06, 08–09, 0F, 11–12, 15, 19–1A, 1E–1F, 27–28, 29, 2A, 2B, 48, 59, 60–61, 64–65, 82–83, 95, 97 	Mathematical Operators (00–FF)
23 	02, 0A, 20–21, 29–2A 	Miscellaneous Technical (00–FF)
25 	00, 02, 0C, 10, 14, 18, 1C, 24, 2C, 34, 3C, 50–6C 	Box Drawing (00–7F)
80, 84, 88, 8C, 90–93 	Block Elements (80–9F)
A0–A1, AA–AC, B2, BA, BC, C4, CA–CB, CF, D8–D9, E6 	Geometric Shapes (A0–FF)
26 	3A–3C, 40, 42, 60, 63, 65–66, 6A, 6B 	Miscellaneous Symbols (00–FF)
F0 	(01–02) 	Private Use Area (00–FF ...)
FB 	01–02 	Alphabetic Presentation Forms (00–4F)
FF 	FD 	Specials

Rendering software that cannot process a Unicode character appropriately often displays it as an open rectangle, or as U+FFFD to indicate the position of the unrecognized character. Some systems have made attempts to provide more information about such characters. Apple's Last Resort font will display a substitute glyph indicating the Unicode range of the character, and the SIL International's fallback font will display a box showing the hexadecimal scalar value of the character.
Mapping and encodings

Several mechanisms have been specified for storing a series of code points as a series of bytes.

Unicode defines two mapping methods: the Unicode Transformation Format (UTF) encodings, and the Universal Coded Character Set (UCS) encodings. An encoding maps (possibly a subset of) the range of Unicode code points to sequences of values in some fixed-size range, termed code units. All UTF encodings map code points to a unique sequence of bytes.[74] The numbers in the names of the encodings indicate the number of bits per code unit (for UTF encodings) or the number of bytes per code unit (for UCS encodings and UTF-1). UTF-8 and UTF-16 are the most commonly used encodings. UCS-2 is an obsolete subset of UTF-16; UCS-4 and UTF-32 are functionally equivalent.

UTF encodings include:

    UTF-8, which uses one to four 8-bit units per code point,[note 3] and has maximal compatibility with ASCII
    UTF-16, which uses one 16-bit unit per code point below U+010000, and a surrogate pair of two 16-bit units per code point in the range U+010000 to U+10FFFF
    UTF-32, which uses one 32-bit unit per code point
    UTF-EBCDIC, not specified as part of The Unicode Standard, which uses one to five 8-bit units per code point, intended to maximize compatibility with EBCDIC

UTF-8 uses one to four 8-bit units (bytes) per code point and, being compact for Latin scripts and ASCII-compatible, provides the de facto standard encoding for the interchange of Unicode text. It is used by FreeBSD and most recent Linux distributions as a direct replacement for legacy encodings in general text handling.

The UCS-2 and UTF-16 encodings specify the Unicode byte order mark (BOM) for use at the beginnings of text files, which may be used for byte-order detection (or byte endianness detection). The BOM, encoded as U+FEFF ZERO WIDTH NO-BREAK SPACE, has the important property of unambiguity on byte reorder, regardless of the Unicode encoding used; U+FFFE (the result of byte-swapping U+FEFF) does not equate to a legal character, and U+FEFF in places other than the beginning of text conveys the zero-width non-break space.

The same character converted to UTF-8 becomes the byte sequence EF BB BF. The Unicode Standard allows the BOM "can serve as a signature for UTF-8 encoded text where the character set is unmarked".[75] Some software developers have adopted it for other encodings, including UTF-8, in an attempt to distinguish UTF-8 from local 8-bit code pages. However RFC 3629, the UTF-8 standard, recommends that byte order marks be forbidden in protocols using UTF-8, but discusses the cases where this may not be possible. In addition, the large restriction on possible patterns in UTF-8 (for instance there cannot be any lone bytes with the high bit set) means that it should be possible to distinguish UTF-8 from other character encodings without relying on the BOM.

In UTF-32 and UCS-4, one 32-bit code unit serves as a fairly direct representation of any character's code point (although the endianness, which varies across different platforms, affects how the code unit manifests as a byte sequence). In the other encodings, each code point may be represented by a variable number of code units. UTF-32 is widely used as an internal representation of text in programs (as opposed to stored or transmitted text), since every Unix operating system that uses the GCC compilers to generate software uses it as the standard "wide character" encoding. Recent versions of the Python programming language (beginning with 2.2) may also be configured to use UTF-32 as the representation for Unicode strings, effectively disseminating such encoding in high-level coded software.

Punycode, another encoding form, enables the encoding of Unicode strings into the limited character set supported by the ASCII-based Domain Name System (DNS). The encoding is used as part of IDNA, which is a system enabling the use of Internationalized Domain Names in all scripts that are supported by Unicode. Earlier and now historical proposals include UTF-5 and UTF-6.

GB18030 is another encoding form for Unicode, from the Standardization Administration of China. It is the official character set of the People's Republic of China (PRC). BOCU-1 and SCSU are Unicode compression schemes. The April Fools' Day RFC of 2005 specified two parody UTF encodings, UTF-9 and UTF-18.
Adoption
See also: UTF-8 § Implementations and adoption
Wikibooks logo
Wikibooks has a book on the topic of: Unicode/Versions

Unicode, in the form of UTF-8, has been the most common encoding for the World Wide Web since 2008.[76] It has near-universal adoption, and much of the non-UTF-8 content is found in other Unicode encodings, e.g. UTF-16. As of 2024, UTF-8 accounts for on average 98.3% of all web pages (and 983 of the top 1,000 highest-ranked web pages).[77] Although many pages only use ASCII characters to display content, UTF-8 was designed with 8-bit ASCII as a subset and almost no websites now declare their encoding to only be ASCII instead of UTF-8.[78] Over a third of the languages tracked have 100% UTF-8 use.

All internet protocols maintained by Internet Engineering Task Force, e.g. File Transfer Protocol (FTP),[79] have required support for UTF-8 since the publication of RFC 2277 in 1998, which specified that all IETF protocols "MUST be able to use the UTF-8 charset".[80]
Operating systems

Unicode has become the dominant scheme for the internal processing and storage of text. Although a great deal of text is still stored in legacy encodings, Unicode is used almost exclusively for building new information processing systems. Early adopters tended to use UCS-2 (the fixed-length two-byte obsolete precursor to UTF-16) and later moved to UTF-16 (the variable-length current standard), as this was the least disruptive way to add support for non-BMP characters. The best known such system is Windows NT (and its descendants, 2000, XP, Vista, 7, 8, 10, and 11), which uses UTF-16 as the sole internal character encoding. The Java and .NET bytecode environments, macOS, and KDE also use it for internal representation. Partial support for Unicode can be installed on Windows 9x through the Microsoft Layer for Unicode.

UTF-8 (originally developed for Plan 9)[81] has become the main storage encoding on most Unix-like operating systems (though others are also used by some libraries) because it is a relatively easy replacement for traditional extended ASCII character sets. UTF-8 is also the most common Unicode encoding used in HTML documents on the World Wide Web.

Multilingual text-rendering engines which use Unicode include Uniscribe and DirectWrite for Microsoft Windows, ATSUI and Core Text for macOS, and Pango for GTK+ and the GNOME desktop.
Input methods
Main article: Unicode input

Because keyboard layouts cannot have simple key combinations for all characters, several operating systems provide alternative input methods that allow access to the entire repertoire.

ISO/IEC 14755,[82] which standardises methods for entering Unicode characters from their code points, specifies several methods. There is the Basic method, where a beginning sequence is followed by the hexadecimal representation of the code point and the ending sequence. There is also a screen-selection entry method specified, where the characters are listed in a table on a screen, such as with a character map program.

Online tools for finding the code point for a known character include Unicode Lookup[83] by Jonathan Hedley and Shapecatcher[84] by Benjamin Milde. In Unicode Lookup, one enters a search key (e.g. "fractions"), and a list of corresponding characters with their code points is returned. In Shapecatcher, based on Shape context, one draws the character in a box and a list of characters approximating the drawing, with their code points, is returned.
Email
Main article: Unicode and email

MIME defines two different mechanisms for encoding non-ASCII characters in email, depending on whether the characters are in email headers (such as the "Subject:"), or in the text body of the message; in both cases, the original character set is identified as well as a transfer encoding. For email transmission of Unicode, the UTF-8 character set and the Base64 or the Quoted-printable transfer encoding are recommended, depending on whether much of the message consists of ASCII characters. The details of the two different mechanisms are specified in the MIME standards and generally are hidden from users of email software.

The IETF has defined[85][86] a framework for internationalized email using UTF-8, and has updated[87][88][89][90] several protocols in accordance with that framework.

The adoption of Unicode in email has been very slow.[citation needed] Some East Asian text is still encoded in encodings such as ISO-2022, and some devices, such as mobile phones,[citation needed] still cannot correctly handle Unicode data. Support has been improving, however. Many major free mail providers such as Yahoo! Mail, Gmail, and Outlook.com support it.
Web
Main article: Unicode and HTML

All W3C recommendations have used Unicode as their document character set since HTML 4.0. Web browsers have supported Unicode, especially UTF-8, for many years. There used to be display problems resulting primarily from font related issues; e.g. v6 and older of Microsoft Internet Explorer did not render many code points unless explicitly told to use a font that contains them.[91]

Although syntax rules may affect the order in which characters are allowed to appear, XML (including XHTML) documents, by definition,[92] comprise characters from most of the Unicode code points, with the exception of:

    FFFE or FFFF.
    most of the C0 control codes,
    the permanently unassigned code points D800–DFFF,

HTML characters manifest either directly as bytes according to the document's encoding, if the encoding supports them, or users may write them as numeric character references based on the character's Unicode code point. For example, the references &#916;, &#1049;, &#1511;, &#1605;, &#3671;, &#12354;, &#21494;, &#33865;, and &#47568; (or the same numeric values expressed in hexadecimal, with &#x as the prefix) should display on all browsers as Δ, Й, ק ,م, ๗, あ, 叶, 葉, and 말.

When specifying URIs, for example as URLs in HTTP requests, non-ASCII characters must be percent-encoded.
Fonts
Main article: Unicode font

Unicode is not in principle concerned with fonts per se, seeing them as implementation choices.[93] Any given character may have many allographs, from the more common bold, italic and base letterforms to complex decorative styles. A font is "Unicode compliant" if the glyphs in the font can be accessed using code points defined in The Unicode Standard.[94] The standard does not specify a minimum number of characters that must be included in the font; some fonts have quite a small repertoire.

Free and retail fonts based on Unicode are widely available, since TrueType and OpenType support Unicode (and Web Open Font Format (WOFF and WOFF2) is based on those). These font formats map Unicode code points to glyphs, but OpenType and TrueType font files are restricted to 65,535 glyphs. Collection files provide a "gap mode" mechanism for overcoming this limit in a single font file. (Each font within the collection still has the 65,535 limit, however.) A TrueType Collection file would typically have a file extension of ".ttc".

Thousands of fonts exist on the market, but fewer than a dozen fonts—sometimes described as "pan-Unicode" fonts—attempt to support the majority of Unicode's character repertoire. Instead, Unicode-based fonts typically focus on supporting only basic ASCII and particular scripts or sets of characters or symbols. Several reasons justify this approach: applications and documents rarely need to render characters from more than one or two writing systems; fonts tend to demand resources in computing environments; and operating systems and applications show increasing intelligence in regard to obtaining glyph information from separate font files as needed, i.e., font substitution. Furthermore, designing a consistent set of rendering instructions for tens of thousands of glyphs constitutes a monumental task; such a venture passes the point of diminishing returns for most typefaces.
Newlines

Unicode partially addresses the newline problem that occurs when trying to read a text file on different platforms. Unicode defines a large number of characters that conforming applications should recognize as line terminators.

In terms of the newline, Unicode introduced U+2028 LINE SEPARATOR and U+2029 PARAGRAPH SEPARATOR. This was an attempt to provide a Unicode solution to encoding paragraphs and lines semantically, potentially replacing all of the various platform solutions. In doing so, Unicode does provide a way around the historical platform-dependent solutions. Nonetheless, few if any Unicode solutions have adopted these Unicode line and paragraph separators as the sole canonical line ending characters. However, a common approach to solving this issue is through newline normalization. This is achieved with the Cocoa text system in macOS and also with W3C XML and HTML recommendations. In this approach, every possible newline character is converted internally to a common newline (which one does not really matter since it is an internal operation just for rendering). In other words, the text system can correctly treat the character as a newline, regardless of the input's actual encoding.
Issues
Character unification
Han unification
Main article: Han unification

The Ideographic Research Group (IRG) is tasked with advising the Consortium and ISO regarding Han unification, or Unihan, especially the further addition of CJK unified and compatibility ideographs to the repertoire. The IRG is composed of experts from each region that has historically used Chinese characters. However, despite the deliberation within the committee, Han unification has consistently been one of the most contested aspects of The Unicode Standard since the genesis of the project.[95]

Existing character set standards such as the Japanese JIS X 0208 (encoded by Shift JIS) defined unification criteria, meaning rules for determining when a variant Chinese character is to be considered a handwriting/font difference (and thus unified), versus a spelling difference (to be encoded separately). Unicode's character model for CJK characters was based on the unification criteria used by JIS X 0208, as well as those developed by the Association for a Common Chinese Code in China.[96]

Due to the standard's principle of encoding semantic instead of stylistic variants, Unicode has received criticism for not assigning code points to certain rare and archaic kanji variants, possibly complicating processing of ancient and uncommon Japanese names. Since it places particular emphasis on Chinese, Japanese and Korean sharing many characters in common, Han unification is also sometimes perceived as treating the three as the same thing.[97] Regional differences in the expected forms of characters, in terms of typographical conventions and curricula for handwriting, do not always fall along language boundaries: although Hong Kong and Taiwan both write Chinese languages using Traditional Chinese characters, the preferred forms of characters differ between Hong Kong and Taiwan in some cases.[98]

Less-frequently-used alternative encodings exist, often predating Unicode, with character models differing from this paradigm, aimed at preserving the various stylistic differences between regional and/or nonstandard character forms. One example is the TRON Code favored by some users for handling historical Japanese text, though not widely adopted among the Japanese public. Another is the CCCII encoding adopted by library systems in Hong Kong, Taiwan and the United States. These have their own drawbacks in general use, leading to the Big5 encoding (introduced in 1984, four years after CCCII) having become more common than CCCII outside of library systems.[99] Although work at Apple based on Research Libraries Group's CJK Thesaurus, which was used to maintain the EACC variant of CCCII, was one of the direct predecessors of Unicode's Unihan set, Unicode adopted the JIS-style unification model.[96]

The earliest version of Unicode had a repertoire of fewer than 21,000 Han characters, largely limited to those in relatively common modern usage. As of version 17.0, the standard now encodes more than 101,000 Han characters, and work is continuing to add thousands more—largely historical and dialectal variant characters used throughout the Sinosphere.

Modern typefaces provide a means to address some of the practical issues in depicting unified Han characters with various regional graphical representations. The 'locl' OpenType table allows a renderer to select a different glyph for each code point based on the text locale.[100] The Unicode variation sequences can also provide in-text annotations for a desired glyph selection; this requires registration of the specific variant in the Ideographic Variation Database.
Italic or cursive characters in Cyrillic
Various Cyrillic characters shown with upright, oblique, and italic alternate forms

If the appropriate glyphs for characters in the same script differ only in the italic, Unicode has generally unified them, as can be seen in the comparison among a set of seven characters' italic glyphs as typically appearing in Russian, traditional Bulgarian, Macedonian, and Serbian texts at right, meaning that the differences are displayed through smart font technology or manually changing fonts. The same OpenType 'locl' technique is used.[101]
Localised case pairs

For use in the Turkish alphabet and Azeri alphabet, Unicode includes a separate dotless lowercase I (ı) and a dotted uppercase I (İ). However, the usual ASCII letters are used for the lowercase dotted i and the uppercase dotless I, matching how they are handled in the earlier ISO 8859-9. As such, case-insensitive comparisons for those languages have to use different rules than case-insensitive comparisons for other languages using the Latin script.[102][103] This can have security implications if, for example, sanitization code or access control relies on case-insensitive comparison.[103]

By contrast, the Icelandic eth (ð), the barred D (đ) and the retroflex D (ɖ), which usually[note 4] look the same in uppercase (Đ), are given the opposite treatment, and encoded separately in both letter-cases (in contrast to the earlier ISO 6937, which unifies the uppercase forms). Although it allows for case-insensitive comparison without needing to know the language of the text, this approach also has issues, requiring security measures relating to homoglyph attacks.[104]
Diacritics on lowercase I
Localised forms of the letter í (I with acute accent)

Whether the lowercase letter I is expected to retain its tittle when a diacritic applies also depends on local conventions.
Security

Unicode has a large number of homoglyphs, many of which look very similar or identical to ASCII letters. Substitution of these can make an identifier or URL that looks correct, but directs to a different location than expected.[105] Additionally, homoglyphs can also be used for manipulating the output of natural language processing (NLP) systems.[106] Mitigation requires disallowing these characters, displaying them differently, or requiring that they resolve to the same identifier;[107] all of this is complicated due to the huge and constantly changing set of characters.[108][109]

A security advisory was released in 2021 by two researchers, one from the University of Cambridge and the other from the University of Edinburgh, in which they assert that the BiDi marks can be used to make large sections of code do something different from what they appear to do. The problem was named "Trojan Source".[110] In response, code editors started highlighting marks to indicate forced text-direction changes.[111]

The UTF-8 and UTF-16 encodings do not accept all possible sequences of code units. Implementations vary in what they do when reading an invalid sequence, which has led to security bugs.[112][113]
Mapping to legacy character sets

Unicode was designed to provide code-point-by-code-point round-trip format conversion to and from any preexisting character encodings, so that text files in older character sets can be converted to Unicode and then back and get back the same file, without employing context-dependent interpretation. That has meant that inconsistent legacy architectures, such as combining diacritics and precomposed characters, both exist in Unicode, giving more than one method of representing some text. This is most pronounced in the three different encoding forms for Korean Hangul. Since version 3.0, any precomposed characters that can be represented by a combined sequence of already existing characters can no longer be added to the standard to preserve interoperability between software using different versions of Unicode.

Injective mappings must be provided between characters in existing legacy character sets and characters in Unicode to facilitate conversion to Unicode and allow interoperability with legacy software. Lack of consistency in various mappings between earlier Japanese encodings such as Shift-JIS or EUC-JP and Unicode led to round-trip format conversion mismatches, particularly the mapping of the character JIS X 0208 '～' (1-33, WAVE DASH), heavily used in legacy database data, to either U+FF5E ～ FULLWIDTH TILDE (in Microsoft Windows) or U+301C 〜 WAVE DASH (other vendors).[114]

Some Japanese computer programmers objected to Unicode because it requires them to separate the use of U+005C \ REVERSE SOLIDUS (backslash) and U+00A5 ¥ YEN SIGN, which was mapped to 0x5C in JIS X 0201, and a lot of legacy code exists with this usage.[115] (This encoding also replaces tilde '~' 0x7E with macron '¯', now 0xAF.) The separation of these characters exists in ISO 8859-1, from long before Unicode.
Indic scripts
Further information: Tamil All Character Encoding

Indic scripts such as Tamil and Devanagari are each allocated only 128 code points, matching the ISCII standard. The correct rendering of Unicode Indic text requires transforming the stored logical order characters into visual order and the forming of ligatures (also known as conjuncts) out of components. Some local scholars argued in favor of assignments of Unicode code points to these ligatures, going against the practice for other writing systems, though Unicode contains some Arabic and other ligatures for backward compatibility purposes only.[116][117][118] Encoding of any new ligatures in Unicode will not happen, in part, because the set of ligatures is font-dependent, and Unicode is an encoding independent of font variations. The same kind of issue arose for the Tibetan script in 2003 when the Standardization Administration of China proposed encoding 956 precomposed Tibetan syllables,[119] but these were rejected for encoding by the relevant ISO committee (ISO/IEC JTC 1/SC 2).[120]

Thai alphabet support has been criticized for its ordering of Thai characters. The vowels เ, แ, โ, ใ, ไ that are written to the left of the preceding consonant are in visual order instead of phonetic order, unlike the Unicode representations of other Indic scripts. This complication is due to Unicode inheriting the Thai Industrial Standard 620, which worked in the same way, and was the way in which Thai had always been written on keyboards. This ordering problem complicates the Unicode collation process slightly, requiring table lookups to reorder Thai characters for collation.[97] Even if Unicode had adopted encoding according to spoken order, it would still be problematic to collate words in dictionary order. E.g., the word แสดง [sa dɛːŋ] "perform" starts with a consonant cluster "สด" (with an inherent vowel for the consonant "ส"), the vowel แ-, in spoken order would come after the ด, but in a dictionary, the word is collated as it is written, with the vowel following the ส.
Combining characters
Main article: Combining character
See also: Unicode normalization § Normalization

Characters with diacritical marks can generally be represented either as a single precomposed character or as a decomposed sequence of a base letter plus one or more non-spacing marks. For example, ḗ (precomposed e with macron and acute above) and ḗ (e followed by the combining macron above and combining acute above) should be rendered identically, both appearing as an e with a macron (◌̄) and acute accent (◌́), but in practice, their appearance may vary depending upon what rendering engine and fonts are being used to display the characters. Similarly, underdots, as needed in the romanization of Indic languages, will often be placed incorrectly.[citation needed] Unicode characters that map to precomposed glyphs can be used in many cases, thus avoiding the problem, but where no precomposed character has been encoded, the problem can often be solved by using a specialist Unicode font such as Charis SIL that uses Graphite, OpenType ('gsub'), or AAT technologies for advanced rendering features.
Anomalies
Main article: Unicode alias names and abbreviations

The Unicode Standard has imposed rules intended to guarantee stability.[121] Depending on the strictness of a rule, a change can be prohibited or allowed. For example, a "name" given to a code point cannot and will not change. But a "script" property is more flexible, by Unicode's own rules. In version 2.0, Unicode changed many code point "names" from version 1. At the same moment, Unicode stated that, thenceforth, an assigned name to a code point would never change. This implies that when mistakes are published, these mistakes cannot be corrected, even if they are trivial (as happened in one instance with the spelling BRAKCET for BRACKET in a character name). In 2006 a list of anomalies in character names was first published, and, as of June 2021, there were 104 characters with identified issues,[122] for example:

    U+034F ͏ COMBINING GRAPHEME JOINER: Does not join graphemes.[122]
    U+2118 ℘ SCRIPT CAPITAL P: This is a small letter. The capital is U+1D4AB 𝒫 MATHEMATICAL SCRIPT CAPITAL P.[123]
    U+A015 ꀕ YI SYLLABLE WU: This is not a Yi syllable, but a Yi iteration mark.
    U+FE18 ︘ PRESENTATION FORM FOR VERTICAL RIGHT WHITE LENTICULAR BRAKCET: bracket is spelled incorrectly.[124] (Spelling errors are resolved by using Unicode alias names.)

While Unicode defines the script designator (name) to be "Phags_Pa", in that script's character names, a hyphen is added: U+A840 ꡀ PHAGS-PA LETTER KA.[125][126] This, however, is not an anomaly, but the rule: hyphens are replaced by underscores in script designators.[125]
See also

    Comparison of Unicode encodings
    International Components for Unicode (ICU), now as ICU-TC a part of Unicode
    List of binary codes
    List of Unicode characters
    List of XML and HTML character entity references
    Lotus Multi-Byte Character Set (LMBCS), a parallel development with similar intentions
    Open-source Unicode typefaces
    Religious and political symbols in Unicode
    Standards related to Unicode
    Unicode symbol
    Universal Coded Character Set

Notes

    "A Unicode Standard Annex (UAX) forms an integral part of The Unicode Standard, but is published as a separate document."
    The two-character prefix U+ was chosen as an ASCII approximation of U+228E ⊎ MULTISET UNION.[62]
    a code point is an abstract representation of an UCS character by an integer between 0 and 1,114,111 (1,114,112 = 220 + 216 or 17 × 216 = 0x110000 code points)
    Rarely, the uppercase Icelandic eth may instead be written in an insular style (Ꝺ) with the crossbar positioned on the stem, particularly if it needs to be distinguished from the uppercase retroflex D (see African Reference Alphabet).

References

    The Unicode Standard, Version 17.0.0. South San Francisco, California: The Unicode Consortium. 2025-09-09. ISBN 978-1-936213-35-1.

    "Unicode Technical Report #28: Unicode 3.2". Unicode Consortium. 2002-03-27. Retrieved 2022-06-23.
    Jenkins, John H. (2021-08-26). "Unicode Standard Annex #45: U-source Ideographs". Unicode Consortium. §2.2 The Source Field. Retrieved 2022-06-23.
        "Unicode Character Count V17.0". The Unicode Consortium. 2025-09-10.
        "Unicode 17.0 Versioned Charts Index". The Unicode Consortium. 2025-09-10.
        "Supported Scripts". The Unicode Consortium. 2025-09-10. Retrieved 2025-09-10.
    "The Unicode Standard: A Technical Introduction". 2019-08-22. Retrieved 2024-09-11.
    "Emoji Counts, v16.0". The Unicode Consortium. Retrieved 2024-09-10.
    Becker, Joseph D. (1998-09-10) [1988-08-29]. "Unicode 88" (PDF). Unicode Consortium. Archived (PDF) from the original on 2016-11-25. Retrieved 2016-10-25. "In 1978, the initial proposal for a set of "Universal Signs" was made by Bob Belleville at Xerox PARC. Many persons contributed ideas to the development of a new encoding design. Beginning in 1980, these efforts evolved into the Xerox Character Code Standard (XCCS) by the present author, a multilingual encoding that has been maintained by Xerox as an internal corporate standard since 1982, through the efforts of Ed Smura, Ron Pellar, and others.
    Unicode arose as the result of eight years of working experience with XCCS. Its fundamental differences from XCCS were proposed by Peter Fenwick and Dave Opstad (pure 16-bit codes) and by Lee Collins (ideographic character unification). Unicode retains the many features of XCCS whose utility has been proved over the years in an international line of communication multilingual system products."
    "Summary Narrative". Unicode. 2006-08-31. Retrieved 2010-03-15.
    "History of Unicode Release and Publication Dates". Unicode. Retrieved 2023-03-20.
    Searle, Stephen J. "Unicode Revisited". Retrieved 2013-01-18.
    "The Unicode Consortium Members". Retrieved 2024-02-12.
    "Unicode Bulldog Award". Unicode. Archived from the original on 2023-11-11.
    "Supported Scripts". Unicode. Retrieved 2025-09-09.
    Otung, Ifiok (2021-01-28). Communication Engineering Principles. John Wiley & Sons. p. 12. ISBN 978-1-119-27407-0.
    "Unicode FAQ". Retrieved 2020-04-02.
    "Roadmap to the BMP". Unicode Consortium. Retrieved 2018-07-30.
    "Roadmaps to Unicode". Unicode. Archived from the original on 2023-12-08.
    "Script Encoding Initiative". Script Encoding Initiative. Archived from the original on 2023-03-25.
    "About The Script Encoding Initiative". The Unicode Consortium. Retrieved 2012-06-04.
    "Scripts to Encode".
    "Unicode 6.1 Paperback Available". announcements_at_unicode.org. Retrieved 2012-05-30.
    "Enumerated Versions of The Unicode Standard". Retrieved 2025-09-12.
        The Unicode Standard, Version 1.0.0. Mountain View, California: The Unicode Consortium. October 1991.
        "1.0.0/UnicodeData.txt (reconstructed)". 2004. Retrieved 2010-03-16.
        The Unicode Standard, Version 1.0.1. Mountain View, California: The Unicode Consortium. June 1992.
        "Unicode Data 1.0.1". Retrieved 2010-03-16.
        The Unicode Standard, Version 1.1.5. Mountain View, California: The Unicode Consortium. July 1995.
        "Unicode Data 1995". Retrieved 2010-03-16.
        The Unicode Standard, Version 2.0.0. Mountain View, California: The Unicode Consortium. July 1996.
        "Unicode Data-2.0.14". Retrieved 2010-03-16.
        The Unicode Standard, Version 2.1.2. Mountain View, California: The Unicode Consortium. May 1998.
        "Unicode Data-2.1.2". Retrieved 2010-03-16.
        The Unicode Standard, Version 3.0.0. Mountain View, California: The Unicode Consortium. September 1999.
        "Unicode Data-3.0.0". Retrieved 2023-10-02.
        The Unicode Standard, Version 3.1.0. Mountain View, California: The Unicode Consortium. March 2001.
        "Unicode Data-3.1.0". Retrieved 2023-10-02.
        The Unicode Standard, Version 3.2.0. Mountain View, California: The Unicode Consortium. March 2002.
        "Unicode Data-3.2.0". Retrieved 2023-10-02.
        The Unicode Standard, Version 4.0.0. Mountain View, California: The Unicode Consortium. April 2003. ISBN 0-321-18578-1.
        "Unicode Data-4.0.0". Retrieved 2023-10-02.
        The Unicode Standard, Version 4.1.0. Mountain View, California: The Unicode Consortium. March 2004. ISBN 0-321-18578-1.
        "Unicode Data-4.1.0". Retrieved 2010-03-16.
    "Named Sequences-4.1.0". Unicode. 2005. Retrieved 2010-03-16.
    The Unicode Standard, Version 5.0.0. Mountain View, California: The Unicode Consortium. 2006-07-14. ISBN 0-321-48091-0.
    "Unicode Data 5.0.0". Retrieved 2010-03-17.
        The Unicode Standard, Version 5.1.0. Mountain View, California: The Unicode Consortium. 2008-04-04. ISBN 0-321-48091-0.
        "Unicode Data 5.1.0". Retrieved 2010-03-17.
        The Unicode Standard, Version 5.2.0. Mountain View, California: The Unicode Consortium. 2009-10-01. ISBN 978-1-936213-00-9.
        "Unicode Data 5.2.0". Retrieved 2010-03-17.
        The Unicode Standard, Version 6.1.0. Mountain View, California: The Unicode Consortium. 2012-01-31. ISBN 978-1-936213-02-3.
        "Unicode Data 6.0.0". Retrieved 2010-10-11.
    "Unicode 6.0 Emoji List". emojipedia.org. Retrieved 2022-09-21.
        The Unicode Standard, Version 6.1.0. Mountain View, California: The Unicode Consortium. 2012-01-31. ISBN 978-1-936213-02-3.
        "Unicode Data 6.1.0". Retrieved 2012-01-31.
        The Unicode Standard, Version 6.2.0. Mountain View, California: The Unicode Consortium. 2012-09-26. ISBN 978-1-936213-07-8.
        "Unicode Data 6.2.0". Retrieved 2012-09-26.
        The Unicode Standard, Version 6.3.0. Mountain View, California: The Unicode Consortium. 2013-09-30. ISBN 978-1-936213-08-5.
        "Unicode Data 6.3.0". Retrieved 2013-09-30.
        The Unicode Standard, Version 7.0.0. Mountain View, California: The Unicode Consortium. 2014-06-16. ISBN 978-1-936213-09-2.
        "Unicode Data 7.0.0". Retrieved 2014-06-15.
        The Unicode Standard, Version 8.0.0. Mountain View, California: The Unicode Consortium. 2015-06-17. ISBN 978-1-936213-10-8.
        "Unicode Data 8.0.0". Retrieved 2015-06-17.
    The Unicode Standard, Version 8.0.0. Mountain View, California: The Unicode Consortium. 2015-06-17. ISBN 978-1-936213-10-8.
    The Unicode Standard, Version 9.0.0. Mountain View, California: The Unicode Consortium. 2016-06-21. ISBN 978-1-936213-13-9.
        The Unicode Standard, Version 9.0.0. Mountain View, California: The Unicode Consortium. 2016-06-21. ISBN 978-1-936213-13-9.
        "Unicode Data 9.0.0". Retrieved 2016-06-21.
    Lobao, Martim (2016-06-07). "These Are The Two Emoji That Weren't Approved For Unicode 9 But Which Google Added To Android Anyway". Android Police. Retrieved 2016-09-04.
    The Unicode Standard, Version 10.0.0. Mountain View, California: The Unicode Consortium. 2017-06-20. ISBN 978-1-936213-16-0.
    The Unicode Standard, Version 11.0.0. Mountain View, California: The Unicode Consortium. 2018-06-05. ISBN 978-1-936213-19-1.
    The Unicode Standard, Version 12.0.0. Mountain View, California: The Unicode Consortium. 2019-03-05. ISBN 978-1-936213-22-1.
    "Unicode Version 12.1 released in support of the Reiwa Era". The Unicode Blog. Retrieved 2019-05-07.
        The Unicode Standard, Version 13.0.0. Mountain View, California: The Unicode Consortium. 2020-03-10. ISBN 978-1-936213-26-9.
        "Announcing The Unicode Standard, Version 13.0". The Unicode Blog. Retrieved 2020-03-11.
    "The Unicode Standard, Version 13.0– Core Specification Appendix C" (PDF). Unicode Consortium. Retrieved 2020-03-11.
        The Unicode Standard, Version 14.0.0. Mountain View, California: The Unicode Consortium. 2021-09-14. ISBN 978-1-936213-29-0.
        "Announcing The Unicode Standard, Version 14.0".
    The Unicode Standard, Version 15.0.0. Mountain View, California: The Unicode Consortium. 2022-09-13. ISBN 978-1-936213-32-0.
        The Unicode Standard, Version 15.1.0. South San Francisco, California: The Unicode Consortium. 2023-09-12. ISBN 978-1-936213-33-7.
    The Unicode Standard, Version 16.0.0. South San Francisco, California: The Unicode Consortium. 2024-09-10. ISBN 978-1-936213-34-4.
    The Unicode Standard, Version 17.0.0. South San Francisco, California: The Unicode Consortium. 2025-09-09. ISBN 978-1-936213-35-1.
    "Glossary of Unicode Terms". Retrieved 2010-03-16.
    "2.4 Code Points and Characters". The Unicode Standard Version 16.0 – Core Specification. 2024.
    "3.4 Characters and Encoding". The Unicode Standard, Version 16.0. 2024.
    "Re: Origin of the U+nnnn notation". Unicode Mail List Archive (Mailing list). 2005-11-08.
    "Appendix A: Notational Conventions". The Unicode Standard. Unicode Consortium. September 2024.
    "Conformance". The Unicode Standard (6.0 ed.). Mountain View, California, US: The Unicode Consortium. 3.9 Unicode Encoding Forms. ISBN 978-1-936213-01-6. "Each encoding form maps the Unicode code points U+0000..U+D7FF and U+E000..U+10FFFF"
    "Unicode Character Encoding Stability Policy". Retrieved 2010-03-16.
    "Properties". Retrieved 2025-09-21.
    "Unicode Character Encoding Model". Retrieved 2023-09-12.
    "Unicode Named Sequences". Retrieved 2025-09-21.
    "Unicode Name Aliases". Retrieved 2025-09-21.
    "JanaSanskritSans". Archived from the original on 2011-07-16.
    CWA 13873:2000 – Multilingual European Subsets in ISO/IEC 10646-1 CEN Workshop Agreement 13873
    Kuhn, Markus (1998). "Multilingual European Character Set 2 (MES-2) Rationale". University of Cambridge. Retrieved 2023-03-20.
    "DIN 91379:2022-08: Characters and defined character sequences in Unicode for the electronic processing of names and data exchange in Europe, with CD-ROM". Beuth Verlag. Retrieved 2022-08-21.
    "UTF-8, UTF-16, UTF-32 & BOM". Unicode.org FAQ. Retrieved 2016-12-12.
    The Unicode Standard, Version 6.2. The Unicode Consortium. 2013. p. 561. ISBN 978-1-936213-08-5.
    Davis, Mark (2008-05-05). "Moving to Unicode 5.1". Official Google Blog. Archived from the original on 2025-04-01. Retrieved 2025-04-12.
    "Usage Survey of Character Encodings broken down by Ranking". W3Techs. Retrieved 2025-04-12.
    "Usage statistics of US-ASCII for websites". W3Techs. Retrieved 2020-11-01.
    B. Curtin (July 1999). Internationalization of the File Transfer Protocol. IETF. doi:10.17487/RFC2640. RFC 2640. Retrieved 2025-04-12.
    H. Alvestrand (January 1998). IETF Policy on Character Sets and Languages. IETF. doi:10.17487/RFC2277. BCP 18. RFC 2277. Archived from the original on 2023-01-23. Retrieved 2025-04-12.
    Pike, Rob (2003-04-30). "UTF-8 history".
    "ISO/IEC JTC1/SC 18/WG 9 N" (PDF). Archived (PDF) from the original on 2025-01-22. Retrieved 2025-04-12.
    Hedley, Jonathan (2009). "Unicode Lookup". Archived from the original on 2025-03-30. Retrieved 2025-04-12.
    Milde, Benjamin (2025). "Unicode Character Recognition". Archived from the original on 2025-04-02.
    J. Klensin; Y. Ko (July 2007). Overview and Framework for Internationalized Email. IETF. doi:10.17487/RFC4952. RFC 4952. Retrieved 2022-08-17.
    J. Klensin; Y. Ko (February 2012). Overview and Framework for Internationalized Email. IETF. doi:10.17487/RFC6530. RFC 6530. Retrieved 2022-08-17.
    J. Yao; W. Mao (February 2012). SMTP Extension for Internationalized Email. IETF. doi:10.17487/RFC6531. RFC 6531. Retrieved 2022-08-17.
    A. Yang; S. Steele; N. Freed (February 2012). Internationalized Email Headers. IETF. doi:10.17487/RFC6532. RFC 6532. Retrieved 2022-08-17.
    C. Newman; A. Gulbrandsen; A. Melnikov (June 2008). Internet Message Access Protocol Internationalization. IETF. doi:10.17487/RFC5255. RFC 5255. Retrieved 2022-08-17.
    R. Gellens; C. Newman (February 2010). POP3 Support for UTF-8. IETF. doi:10.17487/RFC5721. RFC 5721. Retrieved 2022-08-17.
    Wood, Alan (2005-09-13). "Setting up Windows Internet Explorer 5, 5.5 and 6 for Multilingual and Unicode Support: Options for enabling Unicode in Internet Explorer 5, 5.5 and 6: Fonts (IE 5, 5.5 and 6)". Alan Wood. Archived from the original on 2025-01-20. Retrieved 2025-04-12.
    "Extensible Markup Language (XML) 1.1 (Second Edition)". World Wide Web Consortium. 2006-09-29. Archived from the original on 2025-04-05. Retrieved 2025-04-12.
    Bigelow, Charles; Holmes, Kris (September 1993). "The design of a Unicode font" (PDF). Electronic Publishing. 6 (3): 292. ISSN 0894-3982. Archived (PDF) from the original on 2025-02-16. Retrieved 2025-04-12.
    "FAQs: Fonts and keyboards: Fonts and Unicode". Unicode Consortium. Archived from the original on 2025-03-06. Retrieved 2025-04-12.
    A Brief History of Character Codes, Steven J. Searle, originally written 1999, last updated 2004
    "Appendix E: Han Unification History". The Unicode Standard Version 16.0 – Core Specification. Unicode Consortium. 2024.
    Topping, Suzanne (2013-06-25). "The secret life of Unicode". IBM. Archived from the original on 2013-06-25. Retrieved 2023-03-20.
    Lu, Qin (2015-06-08). "The Proposed Hong Kong Character Set" (PDF). ISO/IEC JTC1/SC2/WG2/IRG N2074.
    Wittern, Christian (1995-05-01). "Chinese character codes: an update". International Research Institute for Zen Buddhism / Hanazono University. Archived from the original on 2004-10-12.
    "Noto CJK fonts". Noto Fonts. 2023-02-18. "Select this deployment format if your system supports variable fonts and you prefer to use only one language, but also want full character coverage or the ability to language-tag text to use glyphs that are appropriate for the other languages (this requires an app that supports language tagging and the OpenType 'locl' GSUB feature)."
    Preuss, Ingo. "OpenType Feature: locl – Localized Forms". preusstype.com.
    "Case Folding Properties". Unicode Character Database. Unicode Consortium. 2025-07-30.
    "Regular expression options § Compare using the invariant culture". .NET fundamentals documentation. Microsoft. 2023-05-12.
    "confusablesSummary.txt". Unicode Security Mechanisms for UTS #39. Unicode Consortium. 2023-08-11.
    "UTR #36: Unicode Security Considerations". Unicode.
    Boucher, Nicholas; Shumailov, Ilia; Anderson, Ross; Papernot, Nicolas (2022). "Bad Characters: Imperceptible NLP Attacks". 2022 IEEE Symposium on Security and Privacy (SP). San Francisco, CA, US: IEEE. pp. 1987–2004. arXiv:2106.09898. doi:10.1109/SP46214.2022.9833641. ISBN 978-1-66541-316-9. S2CID 235485405.
    Engineering, Spotify (2013-06-18). "Creative usernames and Spotify account hijacking". Spotify Engineering. Retrieved 2023-04-15.
    Wheeler, David A. (2020). Initial Analysis of Underhanded Source Code (Technical report). p. 4–1–4–10. JSTOR resrep25332.7.
    "UTR #36: Unicode Security Considerations". Unicode. Retrieved 2022-06-27.
    Boucher, Nicholas; Anderson, Ross. "Trojan Source: Invisible Vulnerabilities" (PDF). Retrieved 2021-11-02.
    "Visual Studio Code October 2021". code.visualstudio.com. Retrieved 2021-11-11.
    Dittert, Dominique (2024-09-06). "From Unicode to Exploit: The Security Risks of Overlong UTF-8 Encodings". Retrieved 2024-12-26.
    Boone, Kevin. "UTF-8 and the problem of over-long characters". Retrieved 2024-12-26.
    AFII contribution about WAVE DASH, "An Unicode vendor-specific character table for japanese". 2011-04-22. Archived from the original on 2011-04-22. Retrieved 2019-05-20.
    ISO 646-* Problem Archived 2019-04-23 at the Wayback Machine, Section 4.4.3.5 of Introduction to I18n, Tomohiro Kubota, 2001
    "Arabic Presentation Forms-A" (PDF). Retrieved 2010-03-20.
    "Arabic Presentation Forms-B" (PDF). Retrieved 2010-03-20.
    "Alphabetic Presentation Forms" (PDF). Retrieved 2010-03-20.
    "Proposal on Tibetan BrdaRten Characters Encoding for ISO/IEC 10646 in BMP" (PDF). 2002-12-02.
    Umamaheswaran, V. S. (2003-11-07). "Resolutions of WG 2 meeting 44" (PDF). Resolution M44.20.
    "Character Encoding Stability". Unicode. Archived from the original on 2024-01-01.
    "Unicode Technical Note #27: Known Anomalies in Unicode Character Names". Unicode. 2021-06-14.
    "Unicode chart: "actually this has the form of a lowercase calligraphic p, despite its name"" (PDF).
    "Misspelling of BRACKET in character name is a known defect" (PDF).
    "Unicode Standard Annex #24: Unicode Script Property". The Unicode Consortium. 2021. 2.2 Relation to ISO 15924 Codes. Retrieved 2022-04-29.
    "Scripts.txt". The Unicode Consortium. 2025. Retrieved 2025-09-21.

Further reading

    Julie D. Allen. The Unicode Standard, Version 6.0, The Unicode Consortium, Mountain View, 2011, ISBN 9781936213016, (Unicode 6.0.0).
    The Complete Manual of Typography, James Felici, Adobe Press; 1st edition, 2002. ISBN 0-321-12730-7
    The Unicode Standard, Version 3.0, The Unicode Consortium, Addison-Wesley Longman, Inc., April 2000. ISBN 0-201-61633-5
    The Unicode Standard, Version 4.0, The Unicode Consortium, Addison-Wesley Professional, 27 August 2003. ISBN 0-321-18578-1
    The Unicode Standard, Version 5.0, Fifth Edition, The Unicode Consortium, Addison-Wesley Professional, 27 October 2006. ISBN 0-321-48091-0
    Unicode Demystified: A Practical Programmer's Guide to the Encoding Standard, Richard Gillam, Addison-Wesley Professional; 1st edition, 2002. ISBN 0-201-70052-2
    Unicode Explained, Jukka K. Korpela, O'Reilly; 1st edition, 2006. ISBN 0-596-10121-X
    Unicode: A Primer, Tony Graham, M&T books, 2000. ISBN 0-7645-4625-2.

    Haralambous, Yannis; Martin Dürst (2019). "Unicode from a Linguistic Point of View". In Haralambous, Yannis (ed.). Proceedings of Graphemics in the 21st Century, Brest 2018. Brest: Fluxus Editions. pp. 167–183. doi:10.36824/2018-graf-hara1. ISBN 978-2-9570549-1-6.

External links
Unicode
at Wikipedia's sister projects

    Wikimedia Commons logoMedia from Commons
    Wikibooks logoTextbooks from Wikibooks

    Unicode, Inc.
        Unicode Technical Site
            The Unicode Standard
                Unicode Character Code Charts
                Unicode Character Name Index
    Alan Wood's Unicode Resources – contains lists of word processors with Unicode capability; fonts and characters are grouped by type; characters are presented in lists, not grids.
    Unicode BMP Fallback Font – displays the Unicode 6.1 value of any character in a document, including in the Private Use Area, rather than the glyph itself.
    The World's Writing Systems, all 293 known writing systems with their Unicode status (128 not yet encoded as of June 2024)

    vte

Unicode


	

	

	

	

	

	
	


	

	


	


	

	

	

	


	

	

	

    vte

Character encodings

	

	

	

	

	

	

	

	

	

	

	



	


	

	


	

	

	

	

Authority control databases Edit this at Wikidata
	

	

	

Categories:

    UnicodeCharacter encodingDigital typographyPublic-domain software with source code

"""

<>:436: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:436: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
/tmp/ipykernel_138007/2872650235.py:436: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  Some Japanese computer programmers objected to Unicode because it requires them to separate the use of U+005C \ REVERSE SOLIDUS (backslash) and U+00A5 ¥ YEN SIGN, which was mapped to 0x5C in JIS X 0201, and a lot of legacy code exists with this usage.[115] (This encoding also replaces tilde '~' 0x7E with macron '¯', now 0xAF.) The separation of these characters exists in ISO 8859-1, from long before Unicode.


In [53]:
# A tokenizer contract: encoding followed by decoding must restore the original text.
test_cases = {
    "long Unicode text": valtext,
    "whitespace and code": "def greet(name):\n    return f'Hello, {name}!'\n",
    "emoji": "I like 🍉 and 😄",
    "multilingual text": "hello नमस्ते こんにちは مرحبا",
}

for name, sample in test_cases.items():
    encoded = encode(sample)
    assert decode(encoded) == sample, f"round trip failed: {name}"
    print(f"{name:>20}: {len(sample.encode('utf-8')):>3} bytes -> {len(encoded):>3} tokens")

print("All round-trip tests passed.")

   long Unicode text: 88259 bytes -> 72156 tokens
 whitespace and code:  46 bytes ->  43 tokens
               emoji:  20 bytes ->  17 tokens
   multilingual text:  51 bytes ->  51 tokens
All round-trip tests passed.


# Force splits with regex patterns

Production GPT tokenizers do not run BPE over one entire document. First, they split text into chunks such as words, digits, punctuation, and whitespace. BPE is then trained and applied **inside each chunk only**, so a learned token can never merge across those boundaries. This design makes whitespace and code formatting part of the vocabulary while keeping the tokenizer's behavior predictable.

GPT-4-style tokenizers use a more detailed pattern than GPT-2: contractions are case-insensitive, numbers are grouped in chunks of up to three digits, and newline/whitespace handling is improved.

In [54]:
!pip install regex

In [55]:
import regex as re

gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
gpt4pat = re.compile(r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+""")

sample = "Hello,   world! 123456\n    def greet(name): return 'hi ' + name"
gpt2_chunks = re.findall(gpt2pat, sample)
gpt4_chunks = re.findall(gpt4pat, sample)

assert ''.join(gpt4_chunks) == sample  # pre-tokenization must not lose text
print('GPT-2 chunks:', gpt2_chunks)
print('GPT-4-style chunks:', gpt4_chunks)

GPT-2 chunks: ['Hello', ',', '  ', ' world', '!', ' 123456', '\n   ', ' def', ' greet', '(', 'name', '):', ' return', " '", 'hi', " '", ' +', ' name']
GPT-4-style chunks: ['Hello', ',', '  ', ' world', '!', ' ', '123', '456', '\n', '   ', ' def', ' greet', '(name', '):', ' return', " '", 'hi', " '", ' +', ' name']


In [56]:
!pip install tiktoken

In [57]:
# Inspect the same text with public GPT-2 and GPT-4-era (`cl100k_base`) encodings.
# IDs are tokenizer-specific; the decoded bytes show what each token actually contains.
import tiktoken

comparison_text = "    Hello world! 123456\n    print('नमस्ते')"
for encoding_name in ("gpt2", "cl100k_base"):
    enc = tiktoken.get_encoding(encoding_name)
    token_ids = enc.encode(comparison_text)
    token_bytes = [enc.decode_single_token_bytes(token_id) for token_id in token_ids]
    print(f"\n{encoding_name}: {len(token_ids)} tokens")
    for token_id, piece in zip(token_ids, token_bytes):
        print(f"  {token_id:>6} -> {piece!r}")


gpt2: 27 tokens
     220 -> b' '
     220 -> b' '
     220 -> b' '
   18435 -> b' Hello'
     995 -> b' world'
       0 -> b'!'
   17031 -> b' 123'
   29228 -> b'456'
     198 -> b'\n'
     220 -> b' '
     220 -> b' '
     220 -> b' '
    3601 -> b' print'
   10786 -> b"('"
   11976 -> b'\xe0\xa4'
     101 -> b'\xa8'
   11976 -> b'\xe0\xa4'
     106 -> b'\xae'
   11976 -> b'\xe0\xa4'
     116 -> b'\xb8'
   24231 -> b'\xe0\xa5'
     235 -> b'\x8d'
   11976 -> b'\xe0\xa4'
      97 -> b'\xa4'
   24231 -> b'\xe0\xa5'
     229 -> b'\x87'
   11537 -> b"')"

cl100k_base: 18 tokens
     262 -> b'   '
   22691 -> b' Hello'
    1917 -> b' world'
       0 -> b'!'
     220 -> b' '
    4513 -> b'123'
   10961 -> b'456'
     198 -> b'\n'
     262 -> b'   '
    1194 -> b' print'
     493 -> b"('"
   61196 -> b'\xe0\xa4\xa8'
   88344 -> b'\xe0\xa4\xae'
   79468 -> b'\xe0\xa4\xb8'
   31584 -> b'\xe0\xa5\x8d\xe0\xa4'
      97 -> b'\xa4'
   35470 -> b'\xe0\xa5\x87'
     873 -> b"')"


In [58]:
# `tiktoken` already loads the public GPT-2 merge ranks for us.
# Keeping this notebook API-based makes it reproducible: it no longer depends on
# a manually downloaded `encoder.json` file in the current working directory.

In [59]:
gpt2 = tiktoken.get_encoding("gpt2")
print(f"GPT-2 vocabulary size: {gpt2.n_vocab}")
print("The merge ranks are internal to the encoding object; use `tiktoken` for inference.")

GPT-2 vocabulary size: 50257
The merge ranks are internal to the encoding object; use `tiktoken` for inference.


# Special tokens

Special tokens are reserved tokens with a control meaning rather than ordinary text. For example, GPT-2 uses `<|endoftext|>` to mark the boundary between documents. They are added explicitly to the vocabulary and should be handled deliberately during encoding and decoding so user text cannot accidentally trigger model control behavior.

In [60]:
gpt2.n_vocab  # 256 base bytes + learned BPE tokens + one special token

50257

In [61]:
gpt2.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})  # [50256]

[50256]

# SentencePiece

[SentencePiece](https://github.com/google/sentencepiece) is a widely used tokenizer library. Unlike `tiktoken`'s byte-level BPE approach, it can train and run tokenizers from raw text, and it supports both BPE and the unigram language-model algorithm. SentencePiece tokenizers are used by Llama and Mistral model families.

A key difference is that SentencePiece works over Unicode characters (after normalization) rather than starting from UTF-8 bytes. It also represents whitespace explicitly, commonly with the `▁` marker, so encoding and decoding do not need language-specific pre-tokenization.

`--character_coverage` controls how much of the observed character distribution is retained when building the base character set. Very rare characters outside that coverage normally become the unknown token (`<unk>`). With `--byte_fallback`, SentencePiece can instead decompose an unknown character into UTF-8 byte pieces, preserving its information without adding every rare character to the vocabulary.

**Learning note:** the corpus below is deliberately tiny, so its vocabulary is only an API demonstration—not a useful production tokenizer. Compare its token count with the toy byte-BPE model, but do not treat that comparison as a benchmark.

In [62]:
!pip install sentencepiece

In [63]:
import sentencepiece as spm

In [64]:
from pathlib import Path

artifact_dir = Path("notebook/gpt-tokenizer")
if not artifact_dir.exists():
    artifact_dir = Path(".")  # supports launching Jupyter from this notebook's folder
corpus_path = artifact_dir / "toy.txt"

toy_corpus = """
Tokenizers turn text into token IDs. A useful tokenizer learns from data similar to its future model data.
Python indentation matters: def greet(name): return f'Hello, {name}!'.
Multilingual text matters too: hello नमस्ते こんにちは مرحبا.
Whitespace, punctuation, and emoji like 😄 are part of real text.
"""

with open(corpus_path, "w", encoding="utf-8") as f:
    f.write(toy_corpus)

In [65]:
# train a sentencepiece model on it 
# the settings here are (best effort) those used for training Llama2

import os

options = dict(
  # input spec
  input=str(corpus_path),
  input_format="text",
  # output spec
  model_prefix=str(artifact_dir / "tok400"), # output filename prefix
  # algorithm spec
  # BPE alg
  model_type="bpe",
  vocab_size=400,
  hard_vocab_limit=False, # allow a tiny teaching corpus to produce fewer pieces
  # normalization
  normalization_rule_name="identity", # ew, turn off normalization
  remove_extra_whitespaces=False,
  input_sentence_size=200000000, # max number of training sentences
  max_sentence_length=4192, # max number of bytes per sentence
  seed_sentencepiece_size=1000000,
  shuffle_input_sentence=True,
  # rare word treatment
  character_coverage=0.99995,
  byte_fallback=True,
  # merge rules
  split_digits=True,
  split_by_unicode_script=True,
  split_by_whitespace=True,
  split_by_number=True,
  max_sentencepiece_length=16,
  add_dummy_prefix=True,
  allow_whitespace_only_pieces=True,
  # special tokens
  unk_id=0, # the UNK token MUST exist
  bos_id=1, # the others are optional, set to -1 to turn off
  eos_id=2,
  pad_id=-1,
  # systems
  num_threads=os.cpu_count(), # use ~all system resources
)

spm.SentencePieceTrainer.train(**options)


I0000 00:00:1784649268.272802  138007 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: toy.txt
  input_format: text
  model_prefix: tok400
  model_type: BPE
  vocab_size: 400
  self_test_sample_size: 0
  character_coverage: 0.99995
  input_sentence_size: 200000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_d

True

In [66]:
sp = spm.SentencePieceProcessor()
sp.load(str(artifact_dir / 'tok400.model'))
sp_vocab = [[sp.id_to_piece(idx) , idx ] for idx in range(sp.get_piece_size())]
sp_vocab

[['<unk>', 0],
 ['<s>', 1],
 ['</s>', 2],
 ['<0x00>', 3],
 ['<0x01>', 4],
 ['<0x02>', 5],
 ['<0x03>', 6],
 ['<0x04>', 7],
 ['<0x05>', 8],
 ['<0x06>', 9],
 ['<0x07>', 10],
 ['<0x08>', 11],
 ['<0x09>', 12],
 ['<0x0A>', 13],
 ['<0x0B>', 14],
 ['<0x0C>', 15],
 ['<0x0D>', 16],
 ['<0x0E>', 17],
 ['<0x0F>', 18],
 ['<0x10>', 19],
 ['<0x11>', 20],
 ['<0x12>', 21],
 ['<0x13>', 22],
 ['<0x14>', 23],
 ['<0x15>', 24],
 ['<0x16>', 25],
 ['<0x17>', 26],
 ['<0x18>', 27],
 ['<0x19>', 28],
 ['<0x1A>', 29],
 ['<0x1B>', 30],
 ['<0x1C>', 31],
 ['<0x1D>', 32],
 ['<0x1E>', 33],
 ['<0x1F>', 34],
 ['<0x20>', 35],
 ['<0x21>', 36],
 ['<0x22>', 37],
 ['<0x23>', 38],
 ['<0x24>', 39],
 ['<0x25>', 40],
 ['<0x26>', 41],
 ['<0x27>', 42],
 ['<0x28>', 43],
 ['<0x29>', 44],
 ['<0x2A>', 45],
 ['<0x2B>', 46],
 ['<0x2C>', 47],
 ['<0x2D>', 48],
 ['<0x2E>', 49],
 ['<0x2F>', 50],
 ['<0x30>', 51],
 ['<0x31>', 52],
 ['<0x32>', 53],
 ['<0x33>', 54],
 ['<0x34>', 55],
 ['<0x35>', 56],
 ['<0x36>', 57],
 ['<0x37>', 58],
 ['<0x38>', 5

In [67]:
ids = sp.encode("hello नमस्ते")
print(ids)

[335, 296, 343, 329, 330, 328]


In [68]:
pieces = [sp.id_to_piece(idx) for idx in ids]
print(pieces)

comparison_text = "hello नमस्ते こんにちは 😄"
print(f"SentencePiece: {len(sp.encode(comparison_text))} tokens")
print(f"Toy byte BPE:  {len(encode(comparison_text))} tokens")
print("These counts reflect different tiny training corpora, so compare them only as a learning exercise.")

['▁h', 'ello', '▁', 'नम', 'स्', 'ते']
SentencePiece: 10 tokens
Toy byte BPE:  45 tokens
These counts reflect different tiny training corpora, so compare them only as a learning exercise.


# Building your own GPT-4-style tokenizer

> **Scope:** You can build a tokenizer in the same *family* as GPT-4: byte-level BPE plus GPT-4-style regex pre-tokenization. You cannot reproduce OpenAI's `cl100k_base` exactly unless you have its exact training corpus, merge ranks, regex, and special-token mapping. A tokenizer trained on your own corpus will have different token IDs and must be paired with a model trained using those IDs.

## Build plan

1. **Choose representative data.** Train on the same mix of prose, code, languages, and formatting that your future GPT will see. The tokenizer is trained separately from the LLM, but its data distribution strongly affects context efficiency and multilingual fairness.
2. **Start from all 256 UTF-8 byte values.** This guarantees that every string is encodable, including unseen Unicode characters.
3. **Pre-tokenize with `gpt4pat`.** Split every training document first; count pairs and perform merges only within individual chunks. Never merge across chunks.
4. **Train BPE to a fixed vocabulary size.** Repeatedly merge the most frequent adjacent pair, save `(left_id, right_id) -> new_id`, and build `vocab[new_id] = vocab[left_id] + vocab[right_id]`. Larger vocabularies reduce sequence length but enlarge the embedding and output layers.
5. **Encode with merge ranks.** Pre-tokenize new text, convert each chunk to UTF-8 bytes, then repeatedly apply the available pair with the *lowest* merge ID/rank. Decode by concatenating token bytes first, then calling `decode('utf-8', errors='replace')`. Individual token byte strings may not be valid Unicode by themselves.
6. **Design special tokens deliberately.** Reserve IDs for document boundaries and chat/control markers. Ordinary user text should raise on an unallowed special-token string; silently accepting one is a control-token footgun.
7. **Save the complete tokenizer contract.** Persist the regex pattern, ordered merges, vocabulary, special-token map, normalization policy, and version. Changing any of them changes the meaning of model input IDs.

## Required tests before training a GPT

- Round trip: `decode(encode(text)) == text` for English, code, whitespace, emoji, and multilingual text.
- Regex invariant: joining the pre-tokenized chunks exactly restores the original text.
- Boundary invariant: a merge never crosses two regex chunks.
- Special-token policy: allowed tokens encode intentionally; unallowed ones raise an error.
- Efficiency: compare tokens-per-byte (or tokens-per-character) on held-out data from every target domain and language.

The next practical step is to refactor the toy BPE training loop above so `tokens` becomes a list of UTF-8 byte lists—one list per `gpt4pat` chunk—then train and encode each chunk independently. [Karpathy's `minbpe` reference implementation](https://github.com/karpathy/minbpe) is a compact guide for that refactor.